# Genie Ontology Readiness Assessment (notebook edition)

A notebook version of the **Genie Ontology Readiness** app's *Assess* tab, scoped to one or more catalogs you choose. It uses the app's seven pillars, weights, maturity levels and readiness stages, with no app deployment needed.

**Scoring is coverage-based on focus tables.** These are the gold layer (found by name pattern, layer tag or certification) plus the 25 tables people use most. Scratch and staging schemas are excluded. Each pillar measures how much of that focus set is described, keyed, domain-tagged, certified, stewarded, fresh, readable by business users and actually used. A pillar no longer scores 100 just because a capability exists somewhere. Adoption counts people only; service principals are reported separately. Scores are therefore stricter than the app's.

**Semantic quality is a second axis**, reported next to coverage. It has four checks: description quality (an LLM rubric on a sample), metric consistency (conflicting metric-view measure definitions), column clarity (cryptic names, unexplained codes, missing units, type conflicts), and business vocabulary (Genie question terms against catalog names and synonyms). The LLM is optional: set `use_fmapi = false` and the notebook never calls `ai_query`. Description quality is then not run and the other three checks still score.

**It only reads metadata.** It reads `<catalog>.information_schema` and tags, and, if you turn them on, system tables (`system.access.audit`, `system.access.table_lineage`, `system.query.history`). It never reads table data and never changes anything outside the output location you pick.

| Pillar | Weight | Reads from | Scope |
| --- | --- | --- | --- |
| Unity Catalog Foundation | 15 | `<catalog>.information_schema.tables/schemata` | catalog |
| Metadata Richness | 22 | `information_schema.tables/columns/table_tags` | catalog |
| Relationships & Modeling | 12 | `information_schema.table_constraints`, gold-layer naming | catalog |
| Metrics | 20 | metric views (`table_type = METRIC_VIEW`) | catalog |
| Genie Agents | 16 | `system.access.audit` (default) **or** the Genie REST API | workspace (audit) / catalog (api) |
| Domains & Stewardship | 10 | governed tags + top-accessed tables (`table_lineage`) | catalog |
| Adoption & Activity | 5 | `table_lineage` (default) **or** `audit` + `query.history` | catalog / workspace |

**How to run:** on a fresh copy, run the *Create parameter widgets* cell (or **Run all** once; it stops at the parameter check). The widgets then appear at the top. Fill in `catalogs`, then **Run all** again. If a signal can't be read (missing grant, system tables turned off), that pillar is marked *unavailable* with a reason. The rest of the run keeps going. An unavailable pillar scores 0, the same as in the app.

**Where outputs go** (`output_mode`):
* `auto` (default): writes to every target you filled in (Delta tables, then volume). If you filled in none, or all of them fail, it writes to your workspace home folder instead.
* `table`: appends to `<output_schema>.genie_readiness_runs`, `..._pillars`, `..._drilldown`, and `..._tables` (one row per table with every flag).
* `volume`: writes `report.json/.md/.html`, `drilldown.csv` and `tables.csv` under `<output_volume_path>/<run_id>/`.
* `workspace`: writes the same files under `output_workspace_dir` (default `~/genie_readiness_runs`).
* `display_only`: writes nothing. The report is shown inline, and the full JSON is printed so you can copy it.

**Minimum grants:** `BROWSE` (or `USE CATALOG` + `USE SCHEMA`) on each catalog. For the optional system-table signals, also `SELECT` on `system.access` (and on `system.query` if `adoption_scope = workspace`). Writing outputs needs `CREATE TABLE` on the output schema, or `WRITE VOLUME` on the volume. The workspace and display-only modes need no extra grants.

In [0]:
dbutils.widgets.text("catalogs", "", "01. Catalog(s) to assess (comma-separated)")
dbutils.widgets.dropdown("include_system_tables", "true", ["true", "false"], "02. Use system tables")
dbutils.widgets.dropdown("genie_source", "auto", ["auto", "api", "audit", "skip"], "03. Genie signal source")
dbutils.widgets.dropdown("adoption_scope", "catalog", ["catalog", "workspace"], "04. Adoption scope")
dbutils.widgets.text("lookback_days", "30", "05. Activity lookback (days)")
dbutils.widgets.text("business_principals", "", "06. Business user groups (comma-separated; blank = any non-owner)")
dbutils.widgets.text("exclude_schema_pattern", r"(^|_)(tmp|temp|scratch|sandbox|playground|dev|test|testing|staging|stg|bronze|raw|landing|bak|backup|archive|deprecated)(_|$)", "07. Exclude schemas (regex)")
dbutils.widgets.text("gold_schema_pattern", r"(gold|mart|marts|analytics|semantic|presentation|reporting|dwh)", "08. Gold schema names (regex)")
dbutils.widgets.text("gold_table_pattern", r"^(gold_|mart_|dim_|fact_)", "09. Gold table names (regex)")
dbutils.widgets.text("stale_days", "90", "10. Stale after (days)")
dbutils.widgets.dropdown("output_mode", "auto", ["auto", "table", "volume", "workspace", "display_only"], "11. Output mode")
dbutils.widgets.text("output_schema", "", "12. Output schema (catalog.schema)")
dbutils.widgets.text("output_volume_path", "", "13. Output volume path (/Volumes/c/s/v)")
dbutils.widgets.text("output_workspace_dir", "", "14. Output workspace dir (blank = ~/genie_readiness_runs)")
dbutils.widgets.text("max_genie_spaces", "200", "15. Max Genie spaces to inspect (api mode)")
dbutils.widgets.dropdown("semantic_quality", "true", ["true", "false"], "16. Run semantic quality checks")
dbutils.widgets.dropdown("use_fmapi", "true", ["true", "false"], "17. Use Foundation Model API (ai_query)")
dbutils.widgets.text("llm_endpoint", "databricks-gemini-3-8-flash", "18. LLM endpoint (when FMAPI is on)")
dbutils.widgets.text("llm_sample_size", "40", "19. Objects scored by the LLM")
dbutils.widgets.text("max_genie_conversations", "300", "20. Max Genie conversations for vocabulary")

# This cell only creates the widgets and never fails. Run it once (or Run all) and the
# widgets appear at the top of the notebook; fill in 'catalogs', then Run all again.
print("Widgets ready. Fill in '01. Catalog(s) to assess' at the top, then Run all.")

In [0]:
CATALOGS = [c.strip().strip("`") for c in dbutils.widgets.get("catalogs").split(",") if c.strip()]
INCLUDE_SYSTEM = dbutils.widgets.get("include_system_tables") == "true"
GENIE_SOURCE = dbutils.widgets.get("genie_source")
ADOPTION_SCOPE = dbutils.widgets.get("adoption_scope")
LOOKBACK_DAYS = int(dbutils.widgets.get("lookback_days") or 30)
OUTPUT_MODE = dbutils.widgets.get("output_mode")
OUTPUT_SCHEMA = dbutils.widgets.get("output_schema").strip()
OUTPUT_VOLUME_PATH = dbutils.widgets.get("output_volume_path").strip().rstrip("/")
OUTPUT_WORKSPACE_DIR = dbutils.widgets.get("output_workspace_dir").strip().rstrip("/")
MAX_GENIE_SPACES = int(dbutils.widgets.get("max_genie_spaces") or 200)
BUSINESS_PRINCIPALS = [p.strip().lower() for p in dbutils.widgets.get("business_principals").split(",") if p.strip()]
EXCLUDE_SCHEMA_PATTERN = dbutils.widgets.get("exclude_schema_pattern").strip()
GOLD_SCHEMA_PATTERN = dbutils.widgets.get("gold_schema_pattern").strip()
GOLD_TABLE_PATTERN = dbutils.widgets.get("gold_table_pattern").strip()
STALE_DAYS = int(dbutils.widgets.get("stale_days") or 90)
SEMANTIC_QUALITY = dbutils.widgets.get("semantic_quality") == "true"
USE_FMAPI = dbutils.widgets.get("use_fmapi") == "true"
# With FMAPI off, nothing calls ai_query: description quality is not run and vocabulary uses rule-based filtering only.
LLM_ENDPOINT = dbutils.widgets.get("llm_endpoint").strip() if USE_FMAPI else ""
LLM_SAMPLE_SIZE = max(1, int(dbutils.widgets.get("llm_sample_size") or 40))
MAX_GENIE_CONVERSATIONS = max(0, int(dbutils.widgets.get("max_genie_conversations") or 300))

import re
for _name, _pat in (("exclude_schema_pattern", EXCLUDE_SCHEMA_PATTERN), ("gold_schema_pattern", GOLD_SCHEMA_PATTERN), ("gold_table_pattern", GOLD_TABLE_PATTERN)):
    try:
        re.compile(_pat)
    except re.error as e:
        raise ValueError(f"{_name} is not a valid regular expression: {e}")

if not CATALOGS:
    raise ValueError("The 'catalogs' widget is empty. It is now visible at the top of the notebook: "
                     "enter one or more catalog names (e.g. main or sales,finance) and Run all again.")
if LOOKBACK_DAYS <= 0:
    raise ValueError("lookback_days must be a positive integer.")
print(f"Assessing {CATALOGS} | system tables={INCLUDE_SYSTEM} | genie={GENIE_SOURCE} | adoption={ADOPTION_SCOPE} | "
      f"business users={BUSINESS_PRINCIPALS or 'any non-owner'} | stale after {STALE_DAYS}d | output={OUTPUT_MODE} | "
      f"semantic quality={SEMANTIC_QUALITY} (FMAPI: {LLM_ENDPOINT if USE_FMAPI and LLM_ENDPOINT else 'off'})")

In [0]:
# Pillars, weights, maturity levels and readiness stages mirror app/server/pillars.py.
# Per-pillar scoring is coverage-based on a focus set of tables (gold layer + most-used); see README.
LEVEL_LABELS = ["Absent", "Initial", "Developing", "Established", "Optimized"]

READINESS_STAGES = [
    (0,  "Foundation building",       "Establish Unity Catalog governance and a curated gold layer as the base for Genie."),
    (35, "Core foundation in place",  "Core Unity Catalog governance is in place; strengthen metadata and the semantic layer next."),
    (55, "Semantics and Genie forming", "Metadata and a semantic layer are forming; curate and tune Genie Agents."),
    (72, "Curated and validating",    "Genie Agents are curated; validate accuracy and onboard business users."),
    (85, "Ontology-ready",            "Mature semantics, domains, and adoption. A strong candidate for the learned Genie Ontology preview."),
]

PILLARS = [
    {"key": "uc_foundation", "name": "Unity Catalog Foundation", "weight": 15},
    {"key": "metadata",      "name": "Metadata Richness",        "weight": 22},
    {"key": "relationships", "name": "Relationships & Modeling", "weight": 12},
    {"key": "metrics",       "name": "Metrics",                  "weight": 20},
    {"key": "genie_agents",  "name": "Genie Agents",             "weight": 16},
    {"key": "domains",       "name": "Domains & Stewardship",    "weight": 10},
    {"key": "adoption",      "name": "Adoption & Activity",      "weight": 5},
]

def level_from_score(score):
    if score >= 85: return 4
    if score >= 65: return 3
    if score >= 40: return 2
    if score > 0:   return 1
    return 0

def readiness_stage(score):
    stage = READINESS_STAGES[0]
    for s in READINESS_STAGES:
        if score >= s[0]:
            stage = s
    return {"label": stage[1], "detail": stage[2]}

def readiness_guidance(ranked, fallback):
    names = []
    for p in ranked:
        if p["gaps"] and p["name"] not in names:
            names.append(p["name"])
        if len(names) == 3:
            break
    if not names:
        return fallback
    joined = names[0] if len(names) == 1 else (" and ".join(names) if len(names) == 2 else ", ".join(names[:-1]) + ", and " + names[-1])
    return f"Focus next on {joined}: your lowest-scoring, highest-impact areas."

# Tag keys and heuristics. Edit these to match the customer's conventions.
DOMAIN_TAG_KEYS = ("domain", "data_domain", "business_domain", "subject_area", "data_product")
STEWARD_TAG_KEYS = ("owner", "data_owner", "steward", "data_steward")
CERT_TAG_KEYS = ("system.certification_status", "certification_status")
LAYER_TAG_KEYS = ("layer", "data_layer", "medallion", "medallion_layer", "tier", "data_tier")
GOLD_TAG_VALUES = ("gold", "curated", "presentation", "mart", "semantic", "platinum")
JUNK_TABLE_PATTERN = r"(^|_)(tmp|temp|bak|backup|old|copy|test|scratch)(_|$)|_\d{8}$"
PLACEHOLDER_COMMENTS = ("todo", "tbd", "tbc", "n/a", "na", "none", "null", "-", "--", ".", "description", "comment",
                        "test", "temp", "xxx", "?", "placeholder")
MIN_TABLE_COMMENT_CHARS, MIN_COLUMN_COMMENT_CHARS = 20, 8   # shorter comments count as "thin"
TOP_N_ACCESSED = 25        # most-used tables (by distinct people, 90d) added to the focus set
BROAD_SPACE_TABLES = 25    # Genie Agents above this many tables are flagged as broad (hard limit is 50)
MANAGED_TYPES = ("MANAGED", "MANAGED_SHALLOW_CLONE", "STREAMING_TABLE", "MATERIALIZED_VIEW")

In [0]:
import json, uuid, os, csv, html, time
from datetime import datetime, timezone

def qi(name: str) -> str:
    """Quote an identifier (catalog/schema/table) with backticks."""
    return "`" + str(name).replace("`", "``") + "`"

def ql(value) -> str:
    """Quote a string literal."""
    return "'" + str(value).replace("\\", "\\\\").replace("'", "\\'") + "'"

def in_list(values) -> str:
    return ", ".join(ql(v) for v in values)

_QUERY_LOG = []  # SQL run by the current pillar (kept in the output for explainability)

def run_sql(q: str) -> list:
    _QUERY_LOG.append(" ".join(q.split()))
    return [r.asDict() for r in spark.sql(q).collect()]

def scalar(q: str):
    r = run_sql(q)
    return list(r[0].values())[0] if r else None

def pct(n, d) -> float:
    n, d = float(n or 0), float(d or 0)
    return round(100.0 * n / d, 1) if d else 0.0

def reason_for(exc: Exception) -> str:
    m = str(exc).lower()
    if any(k in m for k in ("permission", "insufficient", "not authorized", "unauthorized", "access denied", "forbidden", "does not have")):
        return "insufficient_permission"
    if any(k in m for k in ("not_found", "not found", "does not exist", "cannot be found", "no such")):
        return "not_enabled"
    return "scan_failed"

def short_err(exc: Exception) -> str:
    return str(exc).strip().splitlines()[0][:300] if str(exc).strip() else type(exc).__name__

def unavailable(note: str, reason: str = None) -> dict:
    return {"available": False, "score": 0.0, "signals": [], "gaps": [], "note": note,
            "metrics": {}, "drill_down": [], "unavailable_reason": reason}

def sig(label, value, detail, unit=None):
    s = {"label": label, "value": value, "detail": detail}
    if unit:
        s["unit"] = unit
    return s

# --- Who / where -------------------------------------------------------------
ASSESSED_BY = spark.sql("SELECT current_user()").collect()[0][0]
WORKSPACE_ID, _w = None, None
try:
    from databricks.sdk import WorkspaceClient
    _w = WorkspaceClient()
    WORKSPACE_ID = str(_w.get_workspace_id())
except Exception as e:
    print(f"Note: could not resolve workspace id via SDK ({short_err(e)}); system-table signals will not be filtered to this workspace.")
WS_PRED = f"AND workspace_id = {ql(WORKSPACE_ID)}" if WORKSPACE_ID else ""

# --- Which requested catalogs are readable ------------------------------------
READABLE, UNREADABLE = [], {}
for c in CATALOGS:
    try:
        spark.sql(f"SELECT 1 FROM {qi(c)}.information_schema.tables LIMIT 1").collect()
        READABLE.append(c)
    except Exception as e:
        UNREADABLE[c] = f"{reason_for(e)}: {short_err(e)}"
if UNREADABLE:
    print("Not readable (skipped):", json.dumps(UNREADABLE, indent=2))
if not READABLE:
    raise RuntimeError("None of the requested catalogs' information_schema is readable. Grant BROWSE or USE CATALOG on them, or check the names.")
READABLE_LC = {c.lower() for c in READABLE}

def src(view: str, alias: str = "_t") -> str:
    """FROM-able union of <catalog>.information_schema.<view> across the assessed catalogs."""
    return "(" + " UNION ALL ".join(f"SELECT * FROM {qi(c)}.information_schema.{view}" for c in READABLE) + f") AS {alias}"

print(f"Readable catalogs: {READABLE} | user: {ASSESSED_BY} | workspace_id: {WORKSPACE_ID}")

In [0]:
import pandas as pd

EXCLUDE_RE = re.compile(EXCLUDE_SCHEMA_PATTERN, re.I) if EXCLUDE_SCHEMA_PATTERN else None
GOLD_S_RE = re.compile(GOLD_SCHEMA_PATTERN, re.I) if GOLD_SCHEMA_PATTERN else None
GOLD_T_RE = re.compile(GOLD_TABLE_PATTERN, re.I) if GOLD_TABLE_PATTERN else None
JUNK_RE = re.compile(JUNK_TABLE_PATTERN, re.I)
INVENTORY_NOTES = []

def fqx(c="table_catalog", s="table_schema", t="table_name"):
    return f"lower(concat_ws('.', {c}, {s}, {t}))"

def meaningful_sql(col, name_col, min_len):
    """A comment counts only if it is long enough, not a placeholder, and not just the object name."""
    return (f"({col} IS NOT NULL AND length(trim({col})) >= {min_len} "
            f"AND lower(trim({col})) NOT IN ({in_list(PLACEHOLDER_COMMENTS)}) "
            f"AND lower(regexp_replace(trim({col}), '[^A-Za-z0-9]', '')) <> lower(regexp_replace({name_col}, '[^A-Za-z0-9]', '')))")

def _df(q, cols):
    return pd.DataFrame([list(r.values()) for r in run_sql(q)], columns=cols)

def _try_df(q, cols, label):
    try:
        return _df(q, cols), True
    except Exception as e:
        INVENTORY_NOTES.append(f"{label} not readable ({reason_for(e)}); related signals are skipped.")
        return pd.DataFrame(columns=cols), False

def _tag_map(df, key, keys):
    d = df[df["k"].isin(keys) & (df["v"].fillna("").astype(str).str.strip() != "")]
    return d.groupby(key)["v"].first().to_dict() if len(d) else {}

def _share(mask, base):
    """Fraction (0-1) of `base` rows where `mask` holds; None when base is empty."""
    n = int(base.sum())
    return int((mask & base).sum()) / n if n else None

def _p(x):
    return "n/a" if x is None else round(100.0 * x, 1)

def parts_score(parts):
    """Weighted 0-100 score from (weight, fraction 0-1 or None). None parts are dropped and the rest re-weighted."""
    usable = [(w, max(0.0, min(1.0, float(v)))) for w, v in parts if v is not None]
    tw = sum(w for w, _ in usable)
    return round(100.0 * sum(w * v for w, v in usable) / tw, 1) if tw else 0.0


def build_inventory():
    INVENTORY_NOTES.clear()
    inv = _df(f"""SELECT {fqx()}, table_catalog, table_schema, table_name, upper(table_type), table_owner,
                    CAST(last_altered AS STRING), comment,
                    CASE WHEN comment IS NOT NULL AND trim(comment) <> '' THEN 1 ELSE 0 END,
                    CASE WHEN {meaningful_sql('comment', 'table_name', MIN_TABLE_COMMENT_CHARS)} THEN 1 ELSE 0 END,
                    CASE WHEN last_altered >= current_timestamp() - INTERVAL {STALE_DAYS} DAYS THEN 1 ELSE 0 END
                  FROM {src('tables')} WHERE table_schema <> 'information_schema'""",
              ["fq", "catalog", "schema", "table", "table_type", "owner", "last_altered", "comment", "has_comment", "comment_ok", "fresh"])
    if inv.empty:
        raise RuntimeError("No tables are visible in the selected catalog(s).")
    cols = _df(f"""SELECT {fqx()}, COUNT(*),
                     SUM(CASE WHEN comment IS NOT NULL AND trim(comment) <> '' THEN 1 ELSE 0 END),
                     SUM(CASE WHEN {meaningful_sql('comment', 'column_name', MIN_COLUMN_COMMENT_CHARS)} THEN 1 ELSE 0 END)
                   FROM {src('columns')} WHERE table_schema <> 'information_schema' GROUP BY 1""",
               ["fq", "n_cols", "n_cols_commented", "n_cols_meaningful"])
    inv = inv.merge(cols, on="fq", how="left")
    for c in ("n_cols", "n_cols_commented", "n_cols_meaningful", "has_comment", "comment_ok", "fresh"):
        inv[c] = pd.to_numeric(inv[c]).fillna(0).astype(int)
    inv["sk"] = (inv["catalog"] + "." + inv["schema"]).str.lower()
    inv["ck"] = inv["catalog"].str.lower()

    # --- Tags: domain (table > schema > catalog), stewardship, certification, layer ---
    tt, _ = _try_df(f"SELECT {fqx('catalog_name', 'schema_name', 'table_name')}, lower(tag_name), tag_value FROM {src('table_tags')}", ["fq", "k", "v"], "Table tags")
    st, _ = _try_df(f"SELECT lower(concat_ws('.', catalog_name, schema_name)), lower(tag_name), tag_value FROM {src('schema_tags')}", ["sk", "k", "v"], "Schema tags")
    ct, _ = _try_df(f"SELECT lower(catalog_name), lower(tag_name), tag_value FROM {src('catalog_tags')}", ["ck", "k", "v"], "Catalog tags")
    inv["domain"] = (inv["fq"].map(_tag_map(tt, "fq", DOMAIN_TAG_KEYS))
                     .fillna(inv["sk"].map(_tag_map(st, "sk", DOMAIN_TAG_KEYS)))
                     .fillna(inv["ck"].map(_tag_map(ct, "ck", DOMAIN_TAG_KEYS))))
    inv["any_tag"] = inv["fq"].isin(set(tt["fq"]))
    inv["steward_tag"] = inv["fq"].isin(set(tt.loc[tt["k"].isin(STEWARD_TAG_KEYS), "fq"])) | \
                         inv["sk"].isin(set(st.loc[st["k"].isin(STEWARD_TAG_KEYS), "sk"]))
    cert = inv["fq"].map(_tag_map(tt, "fq", CERT_TAG_KEYS)).fillna("").astype(str).str.lower()
    inv["certified"], inv["deprecated"] = cert.eq("certified"), cert.eq("deprecated")
    layer = (inv["fq"].map(_tag_map(tt, "fq", LAYER_TAG_KEYS)).fillna(inv["sk"].map(_tag_map(st, "sk", LAYER_TAG_KEYS)))
             .fillna("").astype(str).str.lower())

    # --- Constraints: PK, and FK participation (child or referenced parent) ---
    cons, cons_ok = _try_df(f"SELECT {fqx()}, constraint_type FROM {src('table_constraints')} "
                            f"WHERE constraint_type IN ('PRIMARY KEY', 'FOREIGN KEY')", ["fq", "ct"], "Constraints")
    parents, _ = _try_df(f"""SELECT {fqx('p.table_catalog', 'p.table_schema', 'p.table_name')}
                  FROM {src('referential_constraints', 'r')} JOIN {src('table_constraints', 'p')}
                    ON r.unique_constraint_catalog = p.constraint_catalog AND r.unique_constraint_schema = p.constraint_schema
                   AND r.unique_constraint_name = p.constraint_name""", ["fq"], "FK references")
    inv["has_pk"] = inv["fq"].isin(set(cons.loc[cons["ct"] == "PRIMARY KEY", "fq"]))
    inv["fk_connected"] = inv["fq"].isin(set(cons.loc[cons["ct"] == "FOREIGN KEY", "fq"]) | set(parents["fq"]))

    # --- Access: SELECT (direct or inherited) for business principals, row filters, masks ---
    privs, priv_ok = _try_df(f"SELECT {fqx()}, lower(grantee) FROM {src('table_privileges')} "
                             f"WHERE privilege_type IN ('SELECT', 'ALL_PRIVILEGES')", ["fq", "grantee"], "Table privileges")
    access_known = priv_ok and len(privs) > 0
    if priv_ok and not len(privs):
        INVENTORY_NOTES.append("No grants are visible to you; the access check needs table owner or MANAGE visibility and was skipped.")
    owners = dict(zip(inv["fq"], inv["owner"].fillna("").str.lower()))
    ok_rows = privs[privs["grantee"].isin(BUSINESS_PRINCIPALS)] if BUSINESS_PRINCIPALS else privs[privs["grantee"] != privs["fq"].map(owners)]
    inv["business_access"] = inv["fq"].isin(set(ok_rows["fq"]))
    rf, _ = _try_df(f"SELECT DISTINCT {fqx()} FROM {src('row_filters')}", ["fq"], "Row filters")
    cm, _ = _try_df(f"SELECT {fqx()}, COUNT(*) FROM {src('column_masks')} GROUP BY 1", ["fq", "n"], "Column masks")
    inv["row_filter"] = inv["fq"].isin(set(rf["fq"]))
    inv["masked_cols"] = pd.to_numeric(inv["fq"].map(dict(zip(cm["fq"], cm["n"])))).fillna(0).astype(int)

    # --- Usage: people (identities with '@') vs service principals (UUIDs), from lineage ---
    lineage_known = False
    for c in ("human_users_90d", "sp_users_90d", "human_reads_lb"):
        inv[c] = 0
    if INCLUDE_SYSTEM:
        ln, ok_l = _try_df(f"""SELECT lower(source_table_full_name),
                 COUNT(DISTINCT CASE WHEN created_by LIKE '%@%' THEN created_by END),
                 COUNT(DISTINCT CASE WHEN created_by NOT LIKE '%@%' THEN created_by END),
                 SUM(CASE WHEN created_by LIKE '%@%' AND event_date >= current_date() - INTERVAL {LOOKBACK_DAYS} DAYS THEN 1 ELSE 0 END)
               FROM system.access.table_lineage
               WHERE source_table_catalog IN ({in_list(READABLE)}) AND source_table_full_name IS NOT NULL
                 AND event_date >= current_date() - INTERVAL {max(90, LOOKBACK_DAYS)} DAYS {WS_PRED}
               GROUP BY 1""", ["fq", "human_users_90d", "sp_users_90d", "human_reads_lb"], "Table lineage")
        if ok_l:
            lineage_known = True
            ln = ln.set_index("fq")
            for c in ("human_users_90d", "sp_users_90d", "human_reads_lb"):
                inv[c] = pd.to_numeric(inv["fq"].map(ln[c])).fillna(0).astype(int)

    # --- Scope flags: excluded, clutter, gold, top-accessed, focus ---
    inv["is_metric_view"] = inv["table_type"].isin(["METRIC_VIEW", "METRIC VIEW"])
    inv["is_view"] = inv["table_type"].eq("VIEW") | inv["is_metric_view"]
    inv["excluded"] = inv["schema"].map(lambda s: bool(EXCLUDE_RE.search(s))) if EXCLUDE_RE else False
    inv["junk_name"] = inv["table"].map(lambda t: bool(JUNK_RE.search(t)))
    active = ~inv["excluded"]
    name_spread = inv.loc[active].groupby(inv.loc[active, "table"].str.lower())["sk"].nunique()
    inv["dup_name"] = active & inv["table"].str.lower().map(name_spread).fillna(0).gt(1)
    name_gold = inv["schema"].map(lambda s: bool(GOLD_S_RE and GOLD_S_RE.search(s))) | \
                inv["table"].map(lambda t: bool(GOLD_T_RE and GOLD_T_RE.search(t)))
    inv["is_gold"] = active & ~inv["is_metric_view"] & (name_gold | layer.isin(GOLD_TAG_VALUES) | inv["certified"])
    inv["top_accessed"] = False
    if lineage_known:
        cand = inv[active & ~inv["is_metric_view"] & (inv["human_users_90d"] > 0)].nlargest(TOP_N_ACCESSED, "human_users_90d")
        inv["top_accessed"] = inv["fq"].isin(set(cand["fq"]))
    inv["focus"] = inv["is_gold"] | inv["top_accessed"]
    basis = f"gold layer + top {TOP_N_ACCESSED} most-used tables" if lineage_known else "gold layer"
    if not inv["focus"].any():
        inv["focus"] = active & ~inv["is_metric_view"]
        basis = "all in-scope tables (no gold-layer or frequently used tables detected)"

    owner = inv["owner"].fillna("")
    inv["owner_type"] = owner.map(lambda o: "unknown" if not o else ("person" if "@" in o else "group/SP"))
    inv["stewarded"] = inv["steward_tag"] | inv["owner_type"].eq("group/SP")
    inv["col_meaningful_pct"] = [pct(m, n) for m, n in zip(inv["n_cols_meaningful"], inv["n_cols"])]
    ctx = {"focus_basis": basis, "access_known": access_known, "lineage_known": lineage_known, "constraints_ok": cons_ok,
           "tables": int(len(inv)), "focus": int(inv["focus"].sum()), "excluded": int(inv["excluded"].sum()),
           "gold": int(inv["is_gold"].sum()), "top_accessed": int(inv["top_accessed"].sum())}
    return inv, ctx


INV_COLS = ["fq", "catalog", "schema", "table", "table_type", "owner", "owner_type", "focus", "is_gold", "top_accessed",
            "is_metric_view", "excluded", "junk_name", "dup_name", "comment_ok", "n_cols", "n_cols_meaningful",
            "col_meaningful_pct", "has_pk", "fk_connected", "domain", "any_tag", "stewarded", "certified", "deprecated",
            "business_access", "row_filter", "masked_cols", "fresh", "last_altered", "human_users_90d", "sp_users_90d", "human_reads_lb"]

In [0]:
def _schema_drill(inv, mask, fn):
    out = []
    for (c, s), g in inv[mask].groupby(["catalog", "schema"]):
        out.append({"catalog": c, "schema": s, **fn(g)})
    return out


def probe_uc_foundation():
    inv = INV
    act = ~inv["excluded"]
    data = act & ~inv["is_view"]
    focus_data = inv["focus"] & ~inv["is_view"]
    managed_s = _share(inv["table_type"].isin(MANAGED_TYPES), data)
    fresh_s = _share(inv["fresh"].eq(1), focus_data)
    clutter = inv["excluded"] | inv["junk_name"] | inv["dup_name"]
    clean_s = 1 - float(clutter.mean())
    access_s = _share(inv["business_access"], inv["focus"]) if CTX["access_known"] else None
    score = parts_score([(30, managed_s), (25, fresh_s), (20, clean_s), (25, access_s)])

    who = ", ".join(BUSINESS_PRINCIPALS) if BUSINESS_PRINCIPALS else "anyone other than the owner"
    stale = int((focus_data & inv["fresh"].eq(0)).sum())
    gaps = []
    if managed_s is not None and managed_s < 0.8:
        gaps.append(f"Only {_p(managed_s)}% of in-scope data tables are UC-managed; external and foreign tables miss managed-table optimizations.")
    if fresh_s is not None and fresh_s < 0.8:
        gaps.append(f"{stale} of {int(focus_data.sum())} focus tables haven't changed in {STALE_DAYS}+ days. Confirm they are current, or deprecate them so Genie doesn't answer from stale data.")
    if clutter.mean() > 0.10:
        gaps.append(f"{int(clutter.sum())} of {len(inv)} tables look like clutter (excluded schemas, tmp/backup-style names, or one table name in several schemas). Keep them out of Genie Agents or clean them up.")
    if access_s is not None and access_s < 0.8:
        gaps.append(f"Only {_p(access_s)}% of focus tables are readable by {who}. Genie answers with each user's own permissions, so unreadable tables become failed answers.")
    signals = [sig("In-scope tables", int(act.sum()), f"{CTX['excluded']} excluded by the schema pattern"),
               sig("Focus tables", CTX["focus"], f"Scored set: {CTX['focus_basis']}"),
               sig("Managed", _p(managed_s), "% of in-scope data tables that are UC-managed", "%"),
               sig("Fresh", _p(fresh_s), f"% of focus data tables changed in the last {STALE_DAYS} days", "%"),
               sig("Clutter", int(clutter.sum()), "Excluded, tmp/backup-named or duplicate-named tables")]
    if CTX["access_known"]:
        signals.append(sig("Business-readable", _p(access_s), f"% of focus tables with SELECT for {who}", "%"))
    signals.append(sig("Row filters / masked cols", f"{int(inv['row_filter'].sum())} / {int(inv['masked_cols'].sum())}", "Fine-grained access in place (informational)"))
    note = None if CTX["access_known"] else "Business-access check skipped (grants not visible); score re-weighted across the other signals."
    drill = _schema_drill(inv, inv["table"].notna(), lambda g: {
        "tables": int(len(g)), "excluded": bool(g["excluded"].any()), "focus_tables": int(g["focus"].sum()),
        "managed_pct": pct(int(g.loc[~g["is_view"], "table_type"].isin(MANAGED_TYPES).sum()), int((~g["is_view"]).sum())),
        "stale_focus": int((g["focus"] & ~g["is_view"] & g["fresh"].eq(0)).sum()),
        "clutter": int((g["excluded"] | g["junk_name"] | g["dup_name"]).sum()),
        "business_readable_pct": pct(int((g["focus"] & g["business_access"]).sum()), int(g["focus"].sum()))
                                 if CTX["access_known"] and g["focus"].any() else None})
    drill.sort(key=lambda r: -r["tables"])
    return {"available": True, "score": score, "signals": signals, "gaps": gaps, "note": note, "drill_down": drill,
            "metrics": {"in_scope": int(act.sum()), "excluded": CTX["excluded"], "focus": CTX["focus"], "managed_pct": _p(managed_s),
                        "fresh_pct": _p(fresh_s), "clutter": int(clutter.sum()), "business_readable_pct": _p(access_s)}}


def probe_metadata():
    inv = INV
    f = inv["focus"]
    nf = int(f.sum())
    t_ok = int(inv.loc[f, "comment_ok"].sum())
    c_n, c_ok = int(inv.loc[f, "n_cols"].sum()), int(inv.loc[f, "n_cols_meaningful"].sum())
    weak = int((f & inv["has_comment"].eq(1) & inv["comment_ok"].eq(0)).sum())
    nonmv = ~inv["is_metric_view"]
    raw_t = pct(int(inv.loc[nonmv, "has_comment"].sum()), int(nonmv.sum()))
    raw_c = pct(int(inv.loc[nonmv, "n_cols_commented"].sum()), int(inv.loc[nonmv, "n_cols"].sum()))
    t_s = t_ok / nf if nf else None
    c_s = c_ok / c_n if c_n else None
    score = parts_score([(50, t_s), (50, c_s)])
    gaps = []
    if t_s is not None and t_s < 0.8:
        gaps.append(f"Only {_p(t_s)}% of focus tables have a meaningful description ({nf - t_ok} missing or too thin). Genie relies on these to pick the right table.")
    if c_s is not None and c_s < 0.6:
        gaps.append(f"Only {_p(c_s)}% of focus-table columns have a meaningful comment. Start with columns holding codes, flags, and business terms.")
    if weak:
        gaps.append(f"{weak} focus table(s) have placeholder, very short, or name-only descriptions.")
    signals = [sig("Focus tables described", _p(t_s), f"{t_ok} of {nf} focus tables have a meaningful description", "%"),
               sig("Focus columns described", _p(c_s), f"{c_ok:,} of {c_n:,} focus-table columns", "%"),
               sig("Weak descriptions", weak, f"Placeholder, under {MIN_TABLE_COMMENT_CHARS} chars, or just the table name"),
               sig("All tables commented", raw_t, "Any comment, all tables (the app's measure)", "%"),
               sig("All columns commented", raw_c, "Any comment, all columns (the app's measure)", "%")]
    drill = _schema_drill(inv, f, lambda g: {
        "focus_tables": int(len(g)), "described_pct": pct(int(g["comment_ok"].sum()), int(len(g))),
        "columns": int(g["n_cols"].sum()), "columns_described_pct": pct(int(g["n_cols_meaningful"].sum()), int(g["n_cols"].sum()))})
    drill.sort(key=lambda r: r["described_pct"] + r["columns_described_pct"])
    return {"available": True, "score": score, "signals": signals, "gaps": gaps, "note": f"Scored on focus tables ({CTX['focus_basis']}).",
            "drill_down": drill,
            "metrics": {"focus_tables": nf, "focus_described": t_ok, "focus_columns": c_n, "focus_columns_described": c_ok,
                        "weak_descriptions": weak, "all_tables_commented_pct": raw_t, "all_columns_commented_pct": raw_c}}


def probe_relationships():
    inv = INV
    gold = inv["is_gold"]
    base = gold & ~inv["is_view"]
    n_gold, n_base = int(gold.sum()), int(base.sum())
    note = None
    if not CTX["constraints_ok"]:
        pk_s = conn_s = None
        note = "Constraint metadata not readable; scored on the gold layer only."
    elif n_gold and not n_base:
        pk_s = conn_s = 0.0
        note = "The gold layer is all views; declare PK/FK on the underlying tables, or define joins in metric views."
    else:
        pk_s, conn_s = _share(inv["has_pk"], base), _share(inv["fk_connected"], base)
    score = parts_score([(30, 1.0 if n_gold else 0.0), (35, pk_s if n_gold else 0.0), (35, conn_s if n_gold else 0.0)])
    gaps = []
    if not n_gold:
        gaps.append("No gold layer detected by name, layer tag, or certification. Set gold_schema_pattern / gold_table_pattern, or tag curated schemas (e.g. layer = gold).")
    if pk_s is not None and n_base and pk_s < 0.6:
        gaps.append(f"Only {_p(pk_s)}% of gold tables declare a primary key ({n_base - int((base & inv['has_pk']).sum())} without).")
    if conn_s is not None and n_base and conn_s < 0.5:
        gaps.append(f"Only {_p(conn_s)}% of gold tables take part in a foreign key; declare FKs so Genie can join facts to dimensions reliably.")
    signals = [sig("Gold-layer objects", n_gold, "By name pattern, layer tag, or certification"),
               sig("Gold tables (not views)", n_base, "Tables that can carry constraints")]
    if pk_s is not None:
        signals += [sig("With primary key", _p(pk_s), "% of gold tables", "%"), sig("In a foreign key", _p(conn_s), "% of gold tables, as child or parent", "%")]
    drill = _schema_drill(inv, gold, lambda g: {
        "gold_objects": int(len(g)), "gold_tables": int((~g["is_view"]).sum()),
        "with_pk": int((~g["is_view"] & g["has_pk"]).sum()), "in_fk": int((~g["is_view"] & g["fk_connected"]).sum())})
    drill.sort(key=lambda r: -r["gold_objects"])
    return {"available": True, "score": score, "signals": signals, "gaps": gaps, "note": note, "drill_down": drill,
            "metrics": {"gold_objects": n_gold, "gold_tables": n_base, "pk_pct": _p(pk_s), "fk_pct": _p(conn_s)}}


def probe_metrics():
    inv = INV
    mv = inv["is_metric_view"]
    n = int(mv.sum())
    if n == 0:
        return {"available": True, "score": 0.0, "note": None, "drill_down": [], "metrics": {"metric_views": 0},
                "signals": [sig("Metric views", 0, "UC metric views")],
                "gaps": ["No metric views found. Metric views are the governed metrics layer that feeds Genie Ontology; define core KPIs here."]}
    com_s = _share(inv["comment_ok"].eq(1), mv)
    c_n = int(inv.loc[mv, "n_cols"].sum())
    col_s = int(inv.loc[mv, "n_cols_meaningful"].sum()) / c_n if c_n else None
    used_s = _share(inv["human_users_90d"] > 0, mv) if CTX["lineage_known"] else None
    score = parts_score([(25, 1.0), (15, min(n, 10) / 10), (20, com_s), (25, col_s), (15, used_s)])
    gaps = []
    if n < 3:
        gaps.append("Few metric views; expand coverage so common KPIs are defined once and reused.")
    if com_s is not None and com_s < 1:
        gaps.append(f"{n - int((mv & inv['comment_ok'].eq(1)).sum())} metric view(s) lack a meaningful description.")
    if col_s is not None and col_s < 0.8:
        gaps.append(f"Only {_p(col_s)}% of metric-view dimensions and measures have a meaningful comment; Genie reads these to choose the right measure.")
    if used_s is not None and used_s < 0.5:
        gaps.append(f"Only {_p(used_s)}% of metric views were queried by people in the last 90 days; promote them in Genie Agents and dashboards, or retire unused ones.")
    signals = [sig("Metric views", n, "UC metric views"),
               sig("Described", _p(com_s), "% with a meaningful description", "%"),
               sig("Fields described", _p(col_s), "% of dimensions/measures with a meaningful comment", "%")]
    if used_s is not None:
        signals.append(sig("Used by people", _p(used_s), "% queried by people in the last 90 days", "%"))
    drill = [{"catalog": r["catalog"], "schema": r["schema"], "metric_view": r["table"], "described": bool(r["comment_ok"]),
              "fields": int(r["n_cols"]), "fields_described_pct": float(r["col_meaningful_pct"]), "people_90d": int(r["human_users_90d"])}
             for _, r in inv[mv].sort_values("col_meaningful_pct").iterrows()]
    return {"available": True, "score": score, "signals": signals, "gaps": gaps, "note": None, "drill_down": drill,
            "metrics": {"metric_views": n, "described_pct": _p(com_s), "fields_described_pct": _p(col_s), "used_pct": _p(used_s)}}

In [0]:
GENIE_SPACES = {"scope": None, "ids": []}   # filled by the Genie probe; reused by the vocabulary check
GENIE_Q_ACTIONS = ("geniestartconversationmessage", "geniecreateconversationmessage", "createconversationmessage", "startconversationmessage")
GENIE_FB_ACTIONS = ("geniesendmessagefeedback", "updateconversationmessagefeedback")


def _genie_activity(space_ids=None):
    """Per-space audit activity over the lookback window: questions, people, thumbs up/down (trashed spaces dropped)."""
    space_filter = f"AND request_params.space_id IN ({in_list(space_ids)})" if space_ids else ""
    rating = "upper(coalesce(request_params.rating, request_params.feedback_rating))"
    fb = f"lower(action_name) IN ({in_list(GENIE_FB_ACTIONS)})"
    return run_sql(f"""
        WITH a AS (
          SELECT request_params.space_id AS space_id, COUNT(*) AS events,
                 SUM(CASE WHEN lower(action_name) IN ({in_list(GENIE_Q_ACTIONS)}) THEN 1 ELSE 0 END) AS questions,
                 SUM(CASE WHEN {fb} AND {rating} IN ('NEGATIVE', 'THUMBS_DOWN') THEN 1 ELSE 0 END) AS negative,
                 SUM(CASE WHEN {fb} AND {rating} IN ('POSITIVE', 'THUMBS_UP') THEN 1 ELSE 0 END) AS positive,
                 COUNT(DISTINCT CASE WHEN user_identity.email LIKE '%@%' THEN user_identity.email END) AS users,
                 MAX(CASE WHEN lower(action_name) = 'trashspace' THEN 1 ELSE 0 END) AS trashed,
                 max_by(request_params.display_name, CASE WHEN request_params.display_name IS NOT NULL THEN event_time END) AS name
          FROM system.access.audit
          WHERE service_name = 'aibiGenie' AND request_params.space_id IS NOT NULL AND request_params.space_id <> 'new'
            AND event_date >= current_date() - INTERVAL {LOOKBACK_DAYS} DAYS {WS_PRED} {space_filter}
          GROUP BY request_params.space_id)
        SELECT space_id, COALESCE(name, space_id) AS agent, events, questions, negative, positive, users
        FROM a WHERE trashed = 0 ORDER BY questions DESC, events DESC""")


def _feedback(neg, pos):
    rated = neg + pos
    return (1 - neg / rated) if rated else None


def _genie_audit(prefix_note=""):
    if not INCLUDE_SYSTEM:
        return unavailable(prefix_note + "The audit-log Genie signal needs system tables (include_system_tables = true).", "not_enabled")
    rows = _genie_activity()
    GENIE_SPACES.update(scope="workspace", ids=[r["space_id"] for r in rows if int(r["questions"] or 0) > 0])
    total = len(rows)
    active = sum(1 for r in rows if int(r["questions"] or 0) > 0)
    neg, pos = sum(int(r["negative"] or 0) for r in rows), sum(int(r["positive"] or 0) for r in rows)
    questions = sum(int(r["questions"] or 0) for r in rows)
    fb = _feedback(neg, pos)
    score = parts_score([(40, 1.0 if total else 0.0), (40, active / total if total else None), (20, fb)])
    gaps = []
    if not total:
        gaps.append("No Genie Agents found in the audit log; create a curated Genie Agent over this catalog's gold tables and metric views.")
    elif active / total < 0.5:
        gaps.append(f"{total - active} of {total} Genie Agents had no questions in the last {LOOKBACK_DAYS} days; drive adoption or retire them.")
    if fb is not None and fb < 0.75:
        gaps.append(f"{neg} of {neg + pos} rated answers were thumbs-down; review those conversations and add instructions or example SQL.")
    return {"available": True, "score": score, "gaps": gaps,
            "note": prefix_note + f"Workspace-wide (the audit log doesn't record which tables a space uses). Last {LOOKBACK_DAYS} days.",
            "signals": [sig("Genie Agents", total, "Distinct agents seen in the audit log (workspace-wide)"),
                        sig("Asked questions", active, f"Agents with at least one question in {LOOKBACK_DAYS} days"),
                        sig("Questions", questions, f"Questions asked in {LOOKBACK_DAYS} days"),
                        sig("Thumbs up / down", f"{pos} / {neg}", "Answer feedback")],
            "metrics": {"genie_agents": total, "active": active, "questions": questions, "thumbs_up": pos, "thumbs_down": neg, "scope": "workspace"},
            "drill_down": [{"agent": r["agent"], "space_id": r["space_id"], "questions": int(r["questions"] or 0), "people": int(r["users"] or 0),
                            "thumbs_up": int(r["positive"] or 0), "thumbs_down": int(r["negative"] or 0)} for r in rows[:200]]}


def _genie_api():
    """Catalog-scoped: Genie spaces whose data sources reference the catalog(s). Returns (result, None) or (None, reason)."""
    if _w is None:
        return None, "databricks-sdk is unavailable on this compute"
    spaces, token = [], None
    try:
        while len(spaces) < MAX_GENIE_SPACES:
            q = {"page_size": 100, **({"page_token": token} if token else {})}
            resp = _w.api_client.do("GET", "/api/2.0/genie/spaces", query=q)
            spaces += resp.get("spaces", []) or []
            token = resp.get("next_page_token")
            if not token:
                break
    except Exception as e:
        return None, f"listing spaces failed ({reason_for(e)})"
    spaces = spaces[:MAX_GENIE_SPACES]
    if not spaces:
        return None, "no Genie spaces are visible to you"
    _QUERY_LOG.append(f"REST GET /api/2.0/genie/spaces (+ include_serialized_space per space), {len(spaces)} visible spaces")

    sem_set = set(INV.loc[INV["is_gold"] | INV["is_metric_view"], "fq"])
    in_scope, not_inspectable = [], 0
    for s in spaces:
        try:
            d = _w.api_client.do("GET", f"/api/2.0/genie/spaces/{s['space_id']}", query={"include_serialized_space": "true"})
            ss = d.get("serialized_space")
            ss = json.loads(ss) if isinstance(ss, str) else ss
        except Exception:
            ss = None
        if not isinstance(ss, dict):
            not_inspectable += 1   # reading a space definition needs CAN_EDIT
            continue
        ids = [str(t.get("identifier", "")).replace("`", "").lower() for lst in (ss.get("data_sources") or {}).values()
               if isinstance(lst, list) for t in lst if isinstance(t, dict)]
        hits = [i for i in ids if i.split(".")[0] in READABLE_LC]
        if not hits:
            continue
        ins = ss.get("instructions") or {}
        row = {"agent": s.get("title") or s["space_id"], "space_id": s["space_id"], "tables": len(ids), "tables_in_scope": len(hits),
               "on_gold_or_mv_pct": pct(sum(1 for h in hits if h in sem_set), len(hits)),
               "text_instructions": len(ins.get("text_instructions") or []), "example_sqls": len(ins.get("example_question_sqls") or []),
               "sql_functions": len(ins.get("sql_functions") or []), "benchmarks": len((ss.get("benchmarks") or {}).get("questions") or [])}
        row["curated"] = row["text_instructions"] > 0 and (row["example_sqls"] > 0 or row["sql_functions"] > 0)
        row["broad"] = row["tables"] > BROAD_SPACE_TABLES
        in_scope.append(row)
    if len(spaces) == not_inspectable:
        return None, f"none of the {len(spaces)} visible spaces could be inspected (needs CAN_EDIT)"

    GENIE_SPACES.update(scope="catalog", ids=[r["space_id"] for r in in_scope])
    total = len(in_scope)
    active = neg = pos = None
    act_note = ""
    if INCLUDE_SYSTEM and total:
        try:
            act = {r["space_id"]: r for r in _genie_activity([r["space_id"] for r in in_scope])}
            for r in in_scope:
                a = act.get(r["space_id"], {})
                r["questions"], r["people"] = int(a.get("questions") or 0), int(a.get("users") or 0)
                r["thumbs_up"], r["thumbs_down"] = int(a.get("positive") or 0), int(a.get("negative") or 0)
            active = sum(1 for r in in_scope if r["questions"] > 0)
            neg, pos = sum(r["thumbs_down"] for r in in_scope), sum(r["thumbs_up"] for r in in_scope)
        except Exception as e:
            act_note = f" Activity not readable ({reason_for(e)})."
    curated = sum(1 for r in in_scope if r["curated"])
    benched = sum(1 for r in in_scope if r["benchmarks"] > 0)
    broad = sum(1 for r in in_scope if r["broad"])
    on_sem = (sum(r["on_gold_or_mv_pct"] for r in in_scope) / total / 100) if total else None
    fb = _feedback(neg, pos) if neg is not None else None
    if total:
        score = parts_score([(20, 1.0), (20, active / total if active is not None else None), (20, curated / total),
                             (10, benched / total), (10, 1 - broad / total), (10, on_sem), (10, fb)])
    else:
        score = 0.0
    gaps = []
    if not total:
        gaps.append("No Genie Agents use tables from this catalog; create a curated Genie Agent over its gold tables and metric views.")
    else:
        if active is not None and active / total < 0.5:
            gaps.append(f"{total - active} of {total} Genie Agents on this catalog had no questions in the last {LOOKBACK_DAYS} days.")
        if curated < total:
            gaps.append(f"{total - curated} of {total} Genie Agents lack text instructions plus example SQL or SQL functions.")
        if benched < total:
            gaps.append(f"{total - benched} of {total} Genie Agents have no benchmarks, so answer accuracy isn't being measured.")
        if broad:
            gaps.append(f"{broad} Genie Agent(s) include more than {BROAD_SPACE_TABLES} tables; split into focused agents per domain.")
        if on_sem is not None and on_sem < 0.7:
            gaps.append(f"On average only {_p(on_sem)}% of the tables these agents use are gold tables or metric views.")
        if fb is not None and fb < 0.75:
            gaps.append(f"{neg} of {neg + pos} rated answers were thumbs-down; review those conversations.")
    signals = [sig("Genie Agents on catalog", total, "Spaces whose data sources reference the catalog(s)"),
               sig("Curated", curated, "Instructions + example SQL or SQL functions"),
               sig("With benchmarks", benched, "Have benchmark questions"),
               sig("Broad agents", broad, f"More than {BROAD_SPACE_TABLES} tables"),
               sig("On gold / metric views", _p(on_sem), "Average share of agent tables that are gold or metric views", "%")]
    if active is not None:
        signals += [sig("Asked questions", active, f"Agents with questions in {LOOKBACK_DAYS} days"),
                    sig("Thumbs up / down", f"{pos} / {neg}", "Answer feedback")]
    res = {"available": True, "score": score, "signals": signals, "gaps": gaps, "drill_down": in_scope,
           "note": f"Catalog-scoped via the Genie API. {len(spaces)} space(s) visible, {not_inspectable} not inspectable (needs CAN_EDIT).{act_note}",
           "metrics": {"genie_agents": total, "curated": curated, "benchmarked": benched, "broad": broad, "active": active,
                       "thumbs_up": pos, "thumbs_down": neg, "visible_spaces": len(spaces), "not_inspectable": not_inspectable, "scope": "catalog"}}
    return res, None


def probe_genie_agents():
    GENIE_SPACES.update(scope=None, ids=[])
    if GENIE_SOURCE == "skip":
        return unavailable("Skipped (genie_source = skip).", "not_enabled")
    prefix = ""
    if GENIE_SOURCE in ("api", "auto"):
        res, why = _genie_api()
        if res is not None:
            return res
        if GENIE_SOURCE == "api":
            return unavailable(f"Genie API inspection unavailable: {why}.", "insufficient_permission")
        prefix = f"API inspection unavailable ({why}); fell back to the audit log. "
    return _genie_audit(prefix)


def probe_domains():
    inv = INV
    f = inv["focus"]
    dom_s, stew_s = _share(inv["domain"].notna(), f), _share(inv["stewarded"], f)
    cert_s, tag_s = _share(inv["certified"], f), _share(inv["any_tag"], f)
    n_domains = int(inv.loc[~inv["excluded"], "domain"].dropna().astype(str).str.lower().nunique())
    score = parts_score([(30, dom_s), (25, stew_s), (25, cert_s), (10, min(n_domains, 5) / 5), (10, tag_s)])
    personal = int((f & inv["owner_type"].eq("person") & ~inv["steward_tag"]).sum())
    top = inv[~inv["excluded"] & ~inv["is_metric_view"] & (inv["human_users_90d"] > 0)].nlargest(10, "human_users_90d") if CTX["lineage_known"] else inv.head(0)
    top_cert = int(top["certified"].sum())
    dep_used = inv[inv["deprecated"] & (inv["human_reads_lb"] > 0)]
    gaps = []
    if dom_s is not None and dom_s < 0.8:
        gaps.append(f"Only {_p(dom_s)}% of focus tables roll up to a domain (table, schema, or catalog tag); tag schemas with a domain to cover many tables at once.")
    if stew_s is not None and stew_s < 0.8:
        gaps.append(f"{personal} focus table(s) are owned by an individual with no steward tag; move ownership to a group or tag a steward.")
    if cert_s is not None and cert_s < 0.5:
        gaps.append(f"Only {_p(cert_s)}% of focus tables are certified; certify the canonical gold tables so users and Genie know which to trust.")
    if len(top) and top_cert < len(top):
        gaps.append(f"Only {top_cert} of the top {len(top)} most-used tables are certified; certify high-traffic tables first.")
    if len(dep_used):
        gaps.append(f"{len(dep_used)} deprecated table(s) were still read by people in the last {LOOKBACK_DAYS} days; point users at the replacements.")
    signals = [sig("Distinct domains", n_domains, "Distinct domain tag values (table/schema/catalog)"),
               sig("Focus in a domain", _p(dom_s), "% of focus tables with a domain", "%"),
               sig("Focus stewarded", _p(stew_s), "% owned by a group/SP or carrying a steward tag", "%"),
               sig("Focus certified", _p(cert_s), "% of focus tables certified", "%"),
               sig("Focus tagged", _p(tag_s), "% of focus tables with any tag", "%")]
    if len(top):
        signals.append(sig("Top used certified", top_cert, f"Of the top {len(top)} tables by distinct people (90d)", f"/ {len(top)}"))
    signals.append(sig("Deprecated still used", int(len(dep_used)), f"Deprecated tables read by people in {LOOKBACK_DAYS} days"))
    drill = _schema_drill(inv, f, lambda g: {
        "focus_tables": int(len(g)), "domain": ", ".join(sorted(set(g["domain"].dropna().astype(str))))[:80] or None,
        "in_domain_pct": pct(int(g["domain"].notna().sum()), int(len(g))), "stewarded_pct": pct(int(g["stewarded"].sum()), int(len(g))),
        "certified_pct": pct(int(g["certified"].sum()), int(len(g))), "personal_owner": int((g["owner_type"].eq("person") & ~g["steward_tag"]).sum())})
    drill.sort(key=lambda r: r["in_domain_pct"] + r["certified_pct"])
    return {"available": True, "score": score, "signals": signals, "gaps": gaps, "drill_down": drill,
            "note": "Domains via governed tags; stewardship = group/service-principal owner or a steward tag.",
            "metrics": {"distinct_domains": n_domains, "in_domain_pct": _p(dom_s), "stewarded_pct": _p(stew_s), "certified_pct": _p(cert_s),
                        "tagged_pct": _p(tag_s), "top_used_certified": top_cert, "deprecated_still_used": int(len(dep_used)),
                        "top_used": [{"table": r["fq"], "people_90d": int(r["human_users_90d"]), "certified": bool(r["certified"])} for _, r in top.iterrows()]}}


def _band(users):
    u = users or 0
    return 0.0 if u <= 0 else 20.0 if u < 5 else 35.0 if u < 20 else 45.0 if u < 50 else 50.0


def probe_adoption():
    if not INCLUDE_SYSTEM:
        return unavailable("Skipped (include_system_tables = false).", "not_enabled")
    if ADOPTION_SCOPE == "workspace":
        humans = int(scalar(f"SELECT COUNT(DISTINCT user_identity.email) FROM system.access.audit "
                            f"WHERE user_identity.email LIKE '%@%' AND event_date >= current_date() - INTERVAL {LOOKBACK_DAYS} DAYS {WS_PRED}") or 0)
        queries = int(scalar(f"SELECT COUNT(*) FROM system.query.history WHERE executed_by LIKE '%@%' "
                             f"AND start_time >= current_timestamp() - INTERVAL {LOOKBACK_DAYS} DAYS {WS_PRED}") or 0)
        score = parts_score([(50, _band(humans) / 50), (50, 1.0 if queries else 0.0)])
        gaps = [] if humans else [f"No people active in this workspace in the last {LOOKBACK_DAYS} days."]
        return {"available": True, "score": score, "gaps": gaps, "drill_down": [],
                "note": f"Workspace-wide, people only (service principals excluded), last {LOOKBACK_DAYS} days.",
                "signals": [sig("People active", humans, "Distinct people in the audit log"), sig("Queries by people", queries, "From query history")],
                "metrics": {"people": humans, "queries": queries, "scope": "workspace"}}
    cats = in_list(READABLE)
    r = run_sql(f"""SELECT COUNT(DISTINCT CASE WHEN created_by LIKE '%@%' THEN created_by END) AS humans,
                           COUNT(DISTINCT CASE WHEN created_by NOT LIKE '%@%' THEN created_by END) AS sps,
                           SUM(CASE WHEN created_by LIKE '%@%' THEN 1 ELSE 0 END) AS human_events, COUNT(*) AS events
                    FROM system.access.table_lineage
                    WHERE (source_table_catalog IN ({cats}) OR target_table_catalog IN ({cats}))
                      AND event_date >= current_date() - INTERVAL {LOOKBACK_DAYS} DAYS {WS_PRED}""")[0]
    humans, sps = int(r["humans"] or 0), int(r["sps"] or 0)
    h_ev, ev = int(r["human_events"] or 0), int(r["events"] or 0)
    focus_read = _share(INV["human_reads_lb"] > 0, INV["focus"]) if CTX["lineage_known"] else None
    score = parts_score([(50, _band(humans) / 50), (20, 1.0 if h_ev else 0.0), (30, focus_read)])
    gaps = []
    if not humans:
        gaps.append(f"No people used this catalog in the last {LOOKBACK_DAYS} days" + (f"; only {sps} service principal(s) did." if sps else "."))
    if focus_read is not None and focus_read < 0.5:
        gaps.append(f"Only {_p(focus_read)}% of focus tables were read by people in the last {LOOKBACK_DAYS} days; unused gold tables may be the wrong ones to curate first.")
    return {"available": True, "score": score, "gaps": gaps, "drill_down": [],
            "note": f"Catalog-scoped from lineage, last {LOOKBACK_DAYS} days. People = identities with an email; service principals are reported separately.",
            "signals": [sig("People", humans, "Distinct people reading/writing the catalog"),
                        sig("Service principals", sps, "Automation identities (not scored)"),
                        sig("Events by people", _p(h_ev / ev if ev else None), f"{h_ev:,} of {ev:,} lineage events", "%"),
                        sig("Focus tables used", _p(focus_read), "% of focus tables read by people", "%")],
            "metrics": {"people": humans, "service_principals": sps, "events_by_people": h_ev, "events": ev,
                        "focus_read_pct": _p(focus_read), "scope": "catalog"}}


PROBES = {"uc_foundation": probe_uc_foundation, "metadata": probe_metadata, "relationships": probe_relationships,
          "metrics": probe_metrics, "genie_agents": probe_genie_agents, "domains": probe_domains, "adoption": probe_adoption}

In [0]:
import yaml

# Semantic quality is a second axis, reported next to (not blended into) the coverage score.
SQ_DEFS = [
    {"key": "sq_descriptions",       "name": "Description quality", "weight": 30},
    {"key": "sq_metric_consistency", "name": "Metric consistency",  "weight": 25},
    {"key": "sq_column_clarity",     "name": "Column clarity",      "weight": 25},
    {"key": "sq_vocabulary",         "name": "Business vocabulary", "weight": 20},
]
LLM_FALLBACKS = ("databricks-gemini-3-8-flash", "databricks-glm-5-3", "databricks-meta-llama-3-3-70b-instruct",
                 "databricks-claude-sonnet-4-5", "databricks-gpt-oss-120b")

# --- Heuristics (edit to match the customer's conventions) ------------------------------
KNOWN_ABBR = {"id", "ts", "dt", "qty", "amt", "pct", "num", "cd", "yr", "mo", "wk", "utc", "usd", "gbp", "eur", "sku", "upc",
              "kpi", "sla", "eta", "dob", "url", "ip", "os", "ui", "api", "sql", "geo", "lat", "lng", "lon", "zip", "ytd", "mtd",
              "wtd", "yoy", "mom", "avg", "min", "max", "std", "cnt", "nbr", "no", "tz", "pk", "fk", "src", "dst", "msg", "txn",
              "pos", "crm", "erp", "roi", "aov", "ltv", "cac", "gmv", "nps", "csat", "fte", "bps", "html", "json", "xml", "ddl"}
GENERIC_COLUMN_NAMES = {"status", "type", "code", "flag", "value", "val", "amount", "amt", "name", "desc", "description",
                        "category", "date", "id", "key", "qty", "count", "number", "num", "level", "class", "kind", "group",
                        "ind", "indicator", "data", "info", "misc", "other", "attr", "attribute", "metric", "measure", "total"}
GENERIC_MEASURE_NAMES = {"value", "count", "total", "amount", "metric", "measure", "sum", "avg", "average", "num", "number",
                         "qty", "quantity", "rate", "pct", "percent", "ratio", "cnt"}
CRYPTIC_RE = re.compile(r"^(col|field|column|c|f|attr|var|x|tmp)_?\d+$")
CODE_COL_RE = re.compile(r"(^|_)(cd|code|codes|flag|flg|ind|indicator|type|typ|status|stat|class|cls|cat|category|tier|segment|seg|grade|level|lvl|reason|kind)$")
ENUM_HINT_RE = re.compile(r"\b(one of|values?|possible|either|e\.g\.|such as|codes?|means?|indicates?)\b|\w\s*[=:]\s*\w", re.I)
# Whole-token match (e.g. 'bal' in cannibalization or 'disc' in discipline must not count).
MEASURE_COL_RE = re.compile(r"(^|_)(amount|amt|price|cost|costs|revenue|rev|sales|spend|margin|fee|fees|tax|discount|disc|weight|wt|duration|dur|elapsed|"
                            r"latency|distance|dist|length|height|width|temp|temperature|volume|vol|size|balance|bal|income|profit|gmv|aov|ltv|budget)(_|$)")
NOT_MEASURE_RE = re.compile(r"(_id|_key|_count|_cnt|_qty|_num|_nbr|_flag|_rank|_seq|_code|_cd)$")
UNIT_TOKENS = {"usd", "gbp", "eur", "cad", "aud", "jpy", "inr", "local", "cents", "pct", "percent", "ratio", "kg", "kgs", "g", "gram",
               "grams", "lb", "lbs", "oz", "ms", "sec", "secs", "seconds", "min", "mins", "minutes", "hr", "hrs", "hours", "days", "km",
               "mi", "miles", "m", "meters", "ft", "bytes", "kb", "mb", "gb", "tb", "celsius", "fahrenheit", "bps",
               "dollar", "dollars", "pound", "pounds", "euro", "euros", "pence", "sqft", "sqm", "f", "c", "h", "s", "d",
               "pctg", "secs", "usec", "us", "ns", "in", "inch", "inches", "gal", "gallons", "l", "ml", "liters"}
KNOWN_ABBR |= UNIT_TOKENS   # unit suffixes like _ms, _kg, _f are clear, not cryptic
UNIT_TEXT_RE = re.compile(r"(\$|£|€|%|\b(usd|gbp|eur|dollars?|pounds?|euros?|pence|cents|currency|percent(age)?|ratio|fraction|kg|kilograms?|grams?|"
                          r"lbs?|ounces|seconds|minutes|hours|days|milliseconds|ms|km|kilometers|miles|meters|bytes|mb|gb|celsius|fahrenheit|"
                          r"basis points|bps)\b)", re.I)
VOCAB_STOPWORDS = set("""a an the and or of to in on for by with from at as is are was were be been being this that these those it its
 into per vs versus than then over under between across within without about what which who whom whose when where why how many much
 show me give list tell find get display please can could would should do does did i we our my you your their there here all any each
 every some most more less least top bottom best worst highest lowest total average avg sum count number compare comparison trend trends
 breakdown break down split last next previous prior current past first latest recent today yesterday week weeks weekly month months
 monthly year years yearly quarter quarters quarterly day days daily date dates time period periods ytd mtd yoy also only just like using
 use data table tables query sql chart plot graph rank ranked ranking change changes changed difference increase decrease growth percent
 percentage value values group grouped across vs not no yes have has had will them they other same new old high low based include
 including excluding exclude full overall level levels analysis analyze view see look summary summarize metric metrics
 because often make made makes get got gets give gave go goes going went come came take took put set keep kept stay stays let
 seem seems look looks want wants need needs know think tell try tries ask asks help helps work works run runs end ends start starts
 likely unlikely probably maybe really actually still even ever never always usually sometimes again together among nearby
 something anything nothing everything someone anyone thing things way ways lot lots kind kinds sort type types part parts
 good bad better worse big small large little weak strong reliable different differently similar much many few several
 why whether while during before after since until through around against toward towards upon onto off out up down away back
 chance fix fixed plan plans grow grows fall falls dip dips drop drops rise rises happen happened cause caused reason reasons
 impact impacts affect affects effect effects drive drives driven driving explain explains explained show shows shown
 yeah yes okay hello thanks thank sure right left well just now also either neither both""".split())

RUBRIC_KEYS = ("business_meaning", "grain", "units", "caveats", "column_quality")
RUBRIC_LABELS = {"business_meaning": "business meaning", "grain": "grain", "units": "units", "caveats": "caveats",
                 "column_quality": "column comments"}
RUBRIC_PROMPT = """You review Unity Catalog documentation for Genie, an AI analyst that writes SQL from business questions.
Score how well this <<KIND>> is documented. 0 = missing, 1 = partial, 2 = clear.
- business_meaning: explains what the data represents in business terms, not just restating the name
- grain: states what one row represents (for a metric view: the level its measures aggregate from)
- units: states units, currency or time zone where numeric or date fields need them (null if not applicable)
- caveats: mentions filters, exclusions, known issues, or when not to use it (null if genuinely none are needed)
- column_quality: column comments add business meaning beyond the column name
Also return restates_schema (true if the description mostly repeats names) and issue (the single most useful fix, at most 20 words).
Return only JSON: {"business_meaning": 0, "grain": 0, "units": 0, "caveats": 0, "column_quality": 0, "restates_schema": false, "issue": ""}

Name: <<NAME>>
Description: <<DESCRIPTION>>
Columns (name (type): comment):
<<COLUMNS>>"""
VOCAB_PROMPT = """These words come from questions users asked a data analyst assistant. None of them appear in the catalog's table or
column names, comments, or metric-view synonyms. Classify each word, exactly as written, into one list:
- "concept": a business concept, metric, attribute or entity type the catalog should document or add as a synonym (e.g. churn, sentiment, basket, cannibalization)
- "value": a literal data value such as a specific product, place, person, brand or month name
- "general": an ordinary English word, verb, adjective or fragment that is not a data concept (e.g. often, likely, make, reset)
Return only JSON: {"concept": [...], "value": [...], "general": [...]}.

Words: <<TERMS>>

Sample questions:
<<QUESTIONS>>"""


# --- Shared helpers -----------------------------------------------------------------------
_LLM = {}
def reset_llm():
    _LLM.clear(); _LLM.update(endpoint=None, checked=False, why=None)
reset_llm()

def llm_endpoint():
    """The first chat endpoint that answers ai_query: llm_endpoint, then common fallbacks. None if LLM checks are off."""
    if _LLM["checked"]:
        return _LLM["endpoint"]
    _LLM["checked"] = True
    if not USE_FMAPI:
        _LLM["why"] = "FMAPI disabled (use_fmapi = false)"
        return None
    if not LLM_ENDPOINT:
        _LLM["why"] = "llm_endpoint is blank"
        return None
    for ep in [LLM_ENDPOINT] + [e for e in LLM_FALLBACKS if e != LLM_ENDPOINT]:
        try:
            if spark.sql(f"SELECT ai_query({ql(ep)}, 'Reply with OK') AS r").collect()[0][0]:
                _LLM["endpoint"] = ep
                return ep
        except Exception as e:
            _LLM["why"] = f"{reason_for(e)}: {short_err(e)[:120]}"
    return None

def parse_json(s):
    m = re.search(r"\{.*\}", str(s or ""), re.S)
    if not m:
        return None
    try:
        return json.loads(m.group(0))
    except Exception:
        return None

def llm_batch(prompts: dict) -> dict:
    """{id: prompt} -> {id: parsed JSON or None}, run in parallel with ai_query (prompts go through a DataFrame, not SQL text)."""
    ep = llm_endpoint()
    if not ep or not prompts:
        return {}
    view = f"_gr_prompts_{uuid.uuid4().hex[:8]}"
    spark.createDataFrame(list(prompts.items()), "id string, prompt string").createOrReplaceTempView(view)
    # No modelParameters: several hosted models reject temperature. Per-row errors are kept for the run notes.
    rows = run_sql(f"SELECT id, q.result AS r, q.errorMessage AS err FROM "
                   f"(SELECT id, ai_query({ql(ep)}, prompt, failOnError => false) AS q FROM {view})")
    spark.catalog.dropTempView(view)
    errs = [r["err"] for r in rows if r["err"]]
    _LLM["last_error"] = f"{len(errs)} of {len(rows)} calls failed, e.g. {str(errs[0])[:160]}" if errs else None
    return {r["id"]: parse_json(r["r"]) for r in rows}

def is_meaningful(comment, name, min_len):
    c = str(comment or "").strip()
    if len(c) < min_len or c.lower() in PLACEHOLDER_COMMENTS:
        return False
    return re.sub(r"[^a-z0-9]", "", c.lower()) != re.sub(r"[^a-z0-9]", "", str(name).lower())

def _tokens(text):
    text = re.sub(r"([a-z])([A-Z])", r"\1 \2", str(text or "")).lower().replace("_", " ")
    # Keep contractions whole and drop them ("don't" must not become "don").
    words = re.findall(r"[a-z][a-z0-9]*(?:['’][a-z]+)?", text)
    return [w for w in words if len(w) >= 3 and "'" not in w and "’" not in w]

def _stem(t):
    """Plural-only stemming; -ing/-ed stripping produced fragments like 'someth'. Prefix matching covers the rest."""
    if len(t) > 5 and t.endswith("ies"):
        return t[:-3] + "y"
    if len(t) > 4 and t.endswith(("ses", "xes", "ches", "shes")):
        return t[:-2]
    if len(t) > 3 and t.endswith("s") and not t.endswith(("ss", "us", "is")):
        return t[:-1]
    return t

def _type_family(t):
    t = str(t or "").upper()
    if re.match(r"^(TINYINT|SMALLINT|INT|INTEGER|BIGINT|LONG|SHORT|BYTE)", t): return "INTEGER"
    if re.match(r"^(FLOAT|DOUBLE|REAL|DECIMAL|NUMERIC|DEC)", t): return "NUMERIC"
    if re.match(r"^(STRING|VARCHAR|CHAR)", t): return "STRING"
    if t.startswith("TIMESTAMP"): return "TIMESTAMP"
    return t.split("(")[0].split("<")[0]

def load_focus_columns():
    """Columns (name, type, comment) of focus tables and in-scope metric views."""
    fqs = list(INV.loc[INV["focus"] | (INV["is_metric_view"] & ~INV["excluded"]), "fq"])
    frames = []
    for i in range(0, len(fqs), 400):
        frames.append(_df(f"SELECT {fqx()}, lower(column_name), upper(data_type), comment, ordinal_position FROM {src('columns')} "
                          f"WHERE {fqx()} IN ({in_list(fqs[i:i + 400])})", ["fq", "col", "dtype", "comment", "pos"]))
    cols = pd.concat(frames, ignore_index=True) if frames else pd.DataFrame(columns=["fq", "col", "dtype", "comment", "pos"])
    return cols.merge(INV[["fq", "is_metric_view"]], on="fq", how="left")

def load_metric_views(limit=300):
    """Parsed YAML of in-scope metric views, via DESCRIBE TABLE EXTENDED ... AS JSON (view_text)."""
    out, failed = [], 0
    for _, r in INV[INV["is_metric_view"] & ~INV["excluded"]].head(limit).iterrows():
        try:
            j = json.loads(spark.sql(f"DESCRIBE TABLE EXTENDED {qi(r['catalog'])}.{qi(r['schema'])}.{qi(r['table'])} AS JSON").collect()[0][0])
            y = yaml.safe_load(j.get("view_text") or "") or {}
            out.append({"fq": r["fq"], "yaml": y if isinstance(y, dict) else {}})
        except Exception:
            failed += 1
    return out, failed

def _mv_fields(kind):
    """Flatten metric-view dimensions or measures: one dict per field with its view and source."""
    rows = []
    for m in MVS:
        y = m["yaml"]
        for f in (y.get(kind) or []):
            if isinstance(f, dict) and f.get("name"):
                rows.append({"metric_view": m["fq"], "source": str(y.get("source", "")).lower().replace("`", ""), "name": str(f["name"]),
                             "expr": str(f.get("expr", "")), "comment": f.get("comment"), "display_name": f.get("display_name"),
                             "synonyms": f.get("synonyms") or []})
    return rows


# --- 2. Description quality (LLM rubric on a sample) --------------------------------------
def sq_descriptions():
    ep = llm_endpoint()
    if not ep and not USE_FMAPI:
        return unavailable("Not run: Foundation Model API is disabled (use_fmapi = false). This check needs an LLM; "
                           "the other semantic quality checks are re-weighted without it.", "not_enabled")
    if not ep:
        return unavailable(f"Not run: no usable LLM endpoint for ai_query ({_LLM['why']}). Set llm_endpoint to a chat model you can query, "
                           f"or set use_fmapi = false.", "not_enabled")
    base = INV[INV["has_comment"].eq(1)]
    order = dict(by=["human_users_90d", "fq"], ascending=[False, True])
    tables = base[base["focus"] & ~base["is_metric_view"]].sort_values(**order)
    mvs = base[base["is_metric_view"] & ~base["excluded"]].sort_values(**order)
    n_mv = min(len(mvs), max(1, LLM_SAMPLE_SIZE // 4)) if len(mvs) else 0
    sample = pd.concat([tables.head(LLM_SAMPLE_SIZE - n_mv), mvs.head(n_mv)])
    if sample.empty:
        return {"available": True, "score": 0.0, "signals": [], "metrics": {}, "drill_down": [], "note": None,
                "gaps": ["No focus tables or metric views have a description to evaluate; see Metadata Richness on the coverage axis."]}
    cols_by = {fq: g for fq, g in FOCUS_COLS.groupby("fq")}
    prompts = {}
    for _, r in sample.iterrows():
        g = cols_by.get(r["fq"])
        lines = [] if g is None else [f"- {c['col']} ({c['dtype']}): {str(c['comment'] or '(no comment)').strip()[:150]}"
                                      for _, c in g.sort_values("pos").head(30).iterrows()]
        prompts[r["fq"]] = (RUBRIC_PROMPT.replace("<<KIND>>", "metric view" if r["is_metric_view"] else "table")
                            .replace("<<NAME>>", r["fq"]).replace("<<DESCRIPTION>>", str(r["comment"])[:1500])
                            .replace("<<COLUMNS>>", "\n".join(lines) or "(none)"))
    res = llm_batch(prompts)

    rows = []
    for _, r in sample.iterrows():
        j = res.get(r["fq"])
        if not isinstance(j, dict):
            continue
        vals = {k: j.get(k) for k in RUBRIC_KEYS}
        nums = {k: min(max(float(v), 0.0), 2.0) for k, v in vals.items() if isinstance(v, (int, float)) and not isinstance(v, bool)}
        if not nums:
            continue
        rows.append({"object": r["fq"], "kind": "metric view" if r["is_metric_view"] else "table",
                     "score": round(100 * sum(nums.values()) / (2 * len(nums)), 1), **{k: nums.get(k) for k in RUBRIC_KEYS},
                     "restates_schema": bool(j.get("restates_schema")), "suggested_fix": str(j.get("issue") or "")[:200]})
    if not rows:
        return unavailable(f"The LLM ({ep}) returned no parseable rubric results. {_LLM.get('last_error') or ''}".strip(), "scan_failed")
    df = pd.DataFrame(rows).sort_values("score")
    avg = df["score"].mean() / 100
    def share_full(k):
        s = df[k].dropna()
        return float((s >= 2).mean()) if len(s) else None
    means = {k: df[k].dropna().mean() for k in RUBRIC_KEYS if df[k].notna().any()}
    weakest = min(means, key=means.get) if means else None
    grain_s, units_s, cav_s, col_s = share_full("grain"), share_full("units"), share_full("caveats"), share_full("column_quality")
    restates = int(df["restates_schema"].sum())
    gaps = []
    if avg < 0.7 and weakest:
        gaps.append(f"Sampled descriptions average {_p(avg)}% on the rubric; the weakest area is {RUBRIC_LABELS[weakest]}.")
    if grain_s is not None and grain_s < 0.6:
        gaps.append(f"Only {_p(grain_s)}% of sampled descriptions clearly state the grain (what one row represents); Genie needs this to avoid double counting.")
    if units_s is not None and units_s < 0.6:
        gaps.append(f"Only {_p(units_s)}% clearly state units, currency or time zone where needed.")
    if col_s is not None and col_s < 0.6:
        gaps.append(f"Only {_p(col_s)}% have column comments that add business meaning beyond the column name.")
    if restates:
        gaps.append(f"{restates} sampled description(s) mostly restate the schema; rewrite them in business terms.")
    worst = ", ".join(o.split(".")[-1] for o in df.head(3)["object"])
    if worst and avg < 0.85:
        gaps.append(f"Lowest-rated: {worst}. The drill-down lists a suggested fix for each.")
    signals = [sig("Sampled", len(df), f"Of {len(tables) + len(mvs)} described focus tables and metric views (most-used first)"),
               sig("Rubric score", _p(avg), "Average across business meaning, grain, units, caveats, column comments", "%"),
               sig("Grain stated", _p(grain_s), "% scoring 2 on grain", "%"),
               sig("Units stated", _p(units_s), "% scoring 2 on units (where applicable)", "%"),
               sig("Caveats noted", _p(cav_s), "% scoring 2 on caveats (where applicable)", "%"),
               sig("Restate the schema", restates, "Descriptions that mostly repeat names")]
    failed_note = f" {len(sample) - len(df)} object(s) could not be scored ({_LLM['last_error']})." if len(df) < len(sample) and _LLM.get("last_error") else ""
    return {"available": True, "score": round(100 * avg, 1), "signals": signals, "gaps": gaps,
            "note": f"Scored by {ep} on a sample. Objects without a description are counted on the coverage axis (Metadata Richness), not here.{failed_note}",
            "metrics": {"sampled": len(df), "endpoint": ep, **{f"avg_{k}": round(float(v), 2) for k, v in means.items()}},
            "drill_down": df.to_dict("records")}


# --- 3. Metric consistency (metric-view measure definitions) -------------------------------
def _norm_measure(n):
    n = re.sub(r"[^a-z0-9]+", "_", str(n).lower()).strip("_")
    return re.sub(r"_(total|sum)$", "", re.sub(r"^(total|sum|sum_of)_", "", n))

def _norm_expr(e):
    e = re.sub(r"\bsource\.", "", str(e or "").lower().replace("`", ""))
    return re.sub(r"\s+", "", e)

def sq_metric_consistency():
    if not MVS:
        note = f"{MV_FAILED} metric view definition(s) could not be read." if MV_FAILED else "No metric views in scope."
        return unavailable(note + " Metric consistency needs readable metric-view definitions.", "not_enabled" if not MV_FAILED else "insufficient_permission")
    df = pd.DataFrame(_mv_fields("measures"))
    if df.empty:
        return unavailable("No measures could be parsed from the metric-view definitions.", "scan_failed")
    df["norm"], df["nexpr"] = df["name"].map(_norm_measure), df["expr"].map(_norm_expr)
    df["definition"] = df["source"] + " | " + df["nexpr"]
    by_name = df.groupby("norm").agg(views=("metric_view", "nunique"), definitions=("definition", "nunique"))
    conflicted = by_name[(by_name["views"] > 1) & (by_name["definitions"] > 1)]
    by_formula = df.groupby(["source", "nexpr"]).agg(names=("norm", "nunique"), views=("metric_view", "nunique"))
    dup = by_formula[(by_formula["names"] > 1) & (by_formula["views"] > 1)]
    generic = df[df["norm"].isin(GENERIC_MEASURE_NAMES)]
    n_names, n_formulas = df["norm"].nunique(), len(by_formula)
    score = parts_score([(60, 1 - len(conflicted) / n_names), (20, 1 - len(dup) / n_formulas if n_formulas else None),
                         (20, 1 - len(generic) / len(df))])
    drill = []
    for name in conflicted.sort_values("definitions", ascending=False).index:
        for _, r in df[df["norm"] == name].iterrows():
            drill.append({"issue": "same name, different definition", "measure": name, "metric_view": r["metric_view"],
                          "source": r["source"], "expr": r["expr"][:200]})
    for (srct, nexpr), _ in dup.iterrows():
        for _, r in df[(df["source"] == srct) & (df["nexpr"] == nexpr)].iterrows():
            drill.append({"issue": "same formula, different names", "measure": r["name"], "metric_view": r["metric_view"],
                          "source": r["source"], "expr": r["expr"][:200]})
    for _, r in generic.iterrows():
        drill.append({"issue": "generic measure name", "measure": r["name"], "metric_view": r["metric_view"], "source": r["source"], "expr": r["expr"][:200]})
    gaps = []
    if len(conflicted):
        ex = ", ".join(f"{n} ({int(conflicted.loc[n, 'definitions'])} definitions)" for n in conflicted.sort_values("definitions", ascending=False).index[:3])
        gaps.append(f"{len(conflicted)} measure name(s) are defined differently across metric views, e.g. {ex}. Genie may pick either; "
                    f"rename by grain (e.g. order_revenue vs line_revenue) or consolidate into one definition.")
    if len(dup):
        gaps.append(f"{len(dup)} formula(s) appear under different names in different metric views; pick one canonical name and add the others as synonyms.")
    if len(generic):
        gaps.append(f"{len(generic)} measure(s) have generic names such as {', '.join(sorted(set(generic['name']))[:4])}; name them for what they measure.")
    signals = [sig("Measures", len(df), f"Across {len(MVS)} metric views"), sig("Distinct measure names", n_names, "After removing total_/sum_ prefixes"),
               sig("Conflicting names", len(conflicted), "Same name, different source or formula"),
               sig("Duplicate formulas", len(dup), "Same formula, different names"), sig("Generic names", len(generic), "e.g. value, count, total")]
    note = (f"{MV_FAILED} metric view definition(s) could not be read. " if MV_FAILED else "") + \
           "Compares metric-view measures only; Genie Agent SQL expressions are not yet included."
    return {"available": True, "score": score, "signals": signals, "gaps": gaps, "note": note, "drill_down": drill,
            "metrics": {"measures": len(df), "distinct_names": n_names, "conflicting": len(conflicted), "duplicate_formulas": len(dup), "generic": len(generic)}}


# --- 4. Column clarity (focus-table columns) -----------------------------------------------
def sq_column_clarity():
    c = FOCUS_COLS[FOCUS_COLS["is_metric_view"] != True].copy()
    if c.empty:
        return unavailable("No focus-table columns to evaluate.", "not_enabled")
    c["explained"] = [is_meaningful(cm, n, MIN_COLUMN_COMMENT_CHARS) for cm, n in zip(c["comment"], c["col"])]
    toks = c["col"].map(lambda n: [t for t in re.split(r"[_\d]+", n) if t])
    def cryptic(n, ts):
        if CRYPTIC_RE.match(n) or len(n) <= 2:
            return True
        return any(len(t) <= 4 and not re.search(r"[aeiouy]", t) and t not in KNOWN_ABBR for t in ts)
    c["cryptic"] = [cryptic(n, ts) for n, ts in zip(c["col"], toks)]
    c["generic"] = c["col"].isin(GENERIC_COLUMN_NAMES)
    c["unclear"] = (c["cryptic"] | c["generic"]) & ~c["explained"]
    c["is_code"] = c["col"].map(lambda n: bool(CODE_COL_RE.search(n))) & ~c["dtype"].eq("BOOLEAN")
    c["code_explained"] = c["is_code"] & c["comment"].map(lambda x: bool(x) and bool(ENUM_HINT_RE.search(str(x)) or sum(str(x).count(ch) for ch in ",/|;") >= 2))
    numeric = c["dtype"].map(_type_family).isin(["INTEGER", "NUMERIC"])
    c["is_measure"] = numeric & c["col"].map(lambda n: bool(MEASURE_COL_RE.search(n)) and not NOT_MEASURE_RE.search(n))
    unit_in_name = pd.Series([any(t in UNIT_TOKENS for t in ts) for ts in toks], index=c.index)
    unit_in_comment = c["comment"].map(lambda x: bool(UNIT_TEXT_RE.search(str(x or ""))))
    c["has_unit"] = c["is_measure"] & (unit_in_name | unit_in_comment)
    fam = c.assign(fam=c["dtype"].map(_type_family)).groupby("col").agg(tables=("fq", "nunique"), types=("fam", "nunique"))
    shared = fam[fam["tables"] > 1]
    homonyms = shared[shared["types"] > 1]

    clear_s = 1 - float(c["unclear"].mean())
    code_s = _share(c["code_explained"], c["is_code"])
    unit_s = _share(c["has_unit"], c["is_measure"])
    homo_s = 1 - len(homonyms) / len(shared) if len(shared) else None
    score = parts_score([(30, clear_s), (30, code_s), (25, unit_s), (15, homo_s)])
    def ex(mask, n=5):
        return ", ".join(sorted(set(c.loc[mask, "col"]))[:n])
    gaps = []
    if clear_s < 0.9:
        gaps.append(f"{int(c['unclear'].sum())} focus columns have cryptic or generic names with no explanation (e.g. {ex(c['unclear'])}).")
    if code_s is not None and code_s < 0.6:
        gaps.append(f"Only {_p(code_s)}% of code/flag/status columns explain their values (e.g. {ex(c['is_code'] & ~c['code_explained'])}); list the values and meanings in the comment.")
    if unit_s is not None and unit_s < 0.6:
        gaps.append(f"Only {_p(unit_s)}% of amount/price/duration-style columns state a unit or currency (e.g. {ex(c['is_measure'] & ~c['has_unit'])}).")
    if len(homonyms):
        gaps.append(f"{len(homonyms)} column name(s) have different data types across focus tables (e.g. {', '.join(list(homonyms.index)[:5])}); "
                    f"mismatched join keys and look-alike columns cause wrong joins.")
    signals = [sig("Focus columns", len(c), f"Across {c['fq'].nunique()} focus tables"),
               sig("Clear names", _p(clear_s), "% not cryptic/generic, or explained by a comment", "%"),
               sig("Code columns explained", _p(code_s), f"{int(c['is_code'].sum())} code/flag/status columns", "%"),
               sig("Measures with units", _p(unit_s), f"{int(c['is_measure'].sum())} amount/price/duration-style columns", "%"),
               sig("Type conflicts", len(homonyms), f"Of {len(shared)} column names shared across tables")]
    drill = []
    for fq, g in c.groupby("fq"):
        drill.append({"table": fq, "columns": int(len(g)), "unclear": int(g["unclear"].sum()),
                      "codes_unexplained": int((g["is_code"] & ~g["code_explained"]).sum()),
                      "measures_without_units": int((g["is_measure"] & ~g["has_unit"]).sum()),
                      "examples": ", ".join(list(g.loc[g["unclear"] | (g["is_code"] & ~g["code_explained"]) | (g["is_measure"] & ~g["has_unit"]), "col"])[:6])})
    drill.sort(key=lambda r: -(r["unclear"] + r["codes_unexplained"] + r["measures_without_units"]))
    return {"available": True, "score": score, "signals": signals, "gaps": gaps, "drill_down": drill,
            "note": "Name-based heuristics; adjust KNOWN_ABBR, CODE_COL_RE and MEASURE_COL_RE for the customer's conventions.",
            "metrics": {"columns": len(c), "unclear": int(c["unclear"].sum()), "code_columns": int(c["is_code"].sum()),
                        "codes_explained_pct": _p(code_s), "measure_columns": int(c["is_measure"].sum()), "units_pct": _p(unit_s),
                        "type_conflicts": list(homonyms.index)[:50]}}


# --- 5. Business vocabulary (Genie questions vs catalog vocabulary) -------------------------
MONTH_DAY_WORDS = set("""january february march april june july august september october november december jan feb mar apr
 jun jul aug sep sept oct nov dec monday tuesday wednesday thursday friday saturday sunday mon tue tues wed thu thur thurs fri sat sun""".split())

def _rule_data_value(t):
    """Rule-based data-value filter (works without the LLM): IDs/codes containing digits, and month or day names."""
    return bool(re.search(r"\d", t)) or t in MONTH_DAY_WORDS

def _genie_questions():
    ids = [i for i in GENIE_SPACES["ids"] if re.match(r"^[0-9a-f]{32}$", str(i))]
    if _w is None or not ids or MAX_GENIE_CONVERSATIONS <= 0:
        return []
    per = max(10, MAX_GENIE_CONVERSATIONS // len(ids))
    out = []
    for sid in ids:
        for q in ({"page_size": min(100, per), "include_all": "true"}, {"page_size": min(100, per)}):
            try:
                r = _w.api_client.do("GET", f"/api/2.0/genie/spaces/{sid}/conversations", query=q)
                out += [(sid, str(c.get("title") or "").strip()) for c in (r.get("conversations") or [])[:per] if c.get("title")]
                break
            except Exception:
                continue
        if len(out) >= MAX_GENIE_CONVERSATIONS:
            break
    _QUERY_LOG.append(f"REST GET /api/2.0/genie/spaces/<id>/conversations for {len(ids)} space(s), {len(out)} question(s)")
    return out[:MAX_GENIE_CONVERSATIONS]

def sq_vocabulary():
    # Catalog vocabulary: names, comments and metric-view synonyms of in-scope objects.
    vocab = set()
    act = INV[~INV["excluded"]]
    for text in list(act["table"]) + list(act["domain"].dropna().astype(str)) + list(INV.loc[INV["focus"] | INV["is_metric_view"], "comment"].dropna()):
        vocab.update(_stem(t) for t in _tokens(text))
    for _, r in FOCUS_COLS.iterrows():
        vocab.update(_stem(t) for t in _tokens(r["col"]) + _tokens(r["comment"]))
    fields = _mv_fields("dimensions") + _mv_fields("measures")
    for f in fields:
        for text in [f["name"], f["comment"], f["display_name"]] + [str(s) for s in f["synonyms"]]:
            vocab.update(_stem(t) for t in _tokens(text))
    syn_s = (sum(1 for f in fields if f["synonyms"]) / len(fields)) if fields else None

    questions = _genie_questions()
    df_terms, examples = {}, {}
    for _, q in questions:
        for t in {_stem(w) for w in _tokens(q) if w not in VOCAB_STOPWORDS and not w.isdigit()}:
            df_terms[t] = df_terms.get(t, 0) + 1
            examples.setdefault(t, q[:160])
    min_df = 2 if len(questions) >= 20 else 1
    terms = {t: n for t, n in df_terms.items() if n >= min_df and len(t) >= 3}
    def matched(t):
        return t in vocab or (len(t) >= 5 and any(v.startswith(t[:5]) and abs(len(v) - len(t)) <= 3 for v in vocab))
    status = {t: ("matched" if matched(t) else "data value" if _rule_data_value(t) else "gap") for t in terms}
    llm_used = False
    unmatched = sorted([t for t, s in status.items() if s == "gap"], key=lambda t: -terms[t])[:60]
    if unmatched and llm_endpoint():
        sample_q = "\n".join("- " + q[:200] for _, q in questions[:30])
        j = llm_batch({"vocab": VOCAB_PROMPT.replace("<<TERMS>>", ", ".join(unmatched)).replace("<<QUESTIONS>>", sample_q)}).get("vocab")
        if isinstance(j, dict):
            llm_used = True
            concepts = {str(t).lower() for t in (j.get("concept") or [])}
            for t in unmatched:
                # Only LLM-confirmed business concepts stay as gaps; data values and general words are set aside.
                if t not in concepts:
                    status[t] = "data value" if t in {str(v).lower() for v in (j.get("value") or [])} else "general word"
    counted = {t: n for t, n in terms.items() if status[t] not in ("data value", "general word")}
    total = sum(counted.values())
    cov_s = (sum(n for t, n in counted.items() if status[t] == "matched") / total) if total else None
    score = parts_score([(75, cov_s), (25, syn_s)])
    gap_terms = sorted([t for t in counted if status[t] == "gap"], key=lambda t: -terms[t])
    gaps = []
    if cov_s is not None and cov_s < 0.85 and gap_terms:
        gaps.append(f"Users ask about terms the catalog never mentions: {', '.join(gap_terms[:12])}. "
                    f"Add them as metric-view synonyms, column comments, or glossary terms.")
    if syn_s is not None and syn_s < 0.3:
        gaps.append(f"Only {_p(syn_s)}% of metric-view dimensions and measures have synonyms; add the words business users actually use.")
    if not questions:
        gaps.append("No Genie questions were available to compare against; vocabulary coverage is based on synonyms only.")
    scope = GENIE_SPACES["scope"] or "none"
    signals = [sig("Questions analysed", len(questions), f"Genie conversation titles ({scope} scope)"),
               sig("Terms covered", _p(cov_s), "% of question-term mentions found in names, comments or synonyms", "%"),
               sig("Uncovered terms", len(gap_terms), "Business terms with no match in the catalog"),
               sig("Fields with synonyms", _p(syn_s), f"% of {len(fields)} metric-view dimensions/measures", "%")]
    note = ("Questions come from Genie Agents that use this catalog." if scope == "catalog" else
            "Questions come from all active Genie Agents in the workspace (audit mode), so some terms may belong to other catalogs." if scope == "workspace" else
            "No Genie Agents in scope.")
    note += " IDs, codes and month/day names are set aside as data values."
    note += (" The LLM kept only business concepts as gaps; other data values and general words were set aside." if llm_used else
             " Without the LLM step (FMAPI off), names, places and everyday words may still show up as uncovered terms.")
    drill = [{"term": t, "questions": terms[t], "status": status[t], "example": examples.get(t, "")}
             for t in sorted(terms, key=lambda t: (status[t] != "gap", -terms[t]))[:200]]
    return {"available": True, "score": score, "signals": signals, "gaps": gaps, "note": note, "drill_down": drill,
            "metrics": {"questions": len(questions), "terms": len(terms), "covered_pct": _p(cov_s), "uncovered": gap_terms[:50],
                        "synonym_pct": _p(syn_s), "question_scope": scope}}


SQ_PROBES = {"sq_descriptions": sq_descriptions, "sq_metric_consistency": sq_metric_consistency,
             "sq_column_clarity": sq_column_clarity, "sq_vocabulary": sq_vocabulary}

In [0]:
RUN_ID = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ") + "_" + uuid.uuid4().hex[:6]
ASSESSED_AT = datetime.now(timezone.utc)

_QUERY_LOG.clear()
_t0 = time.time()
INV, CTX = build_inventory()
INVENTORY_QUERIES = list(_QUERY_LOG)
print(f"Inventory: {CTX['tables']:,} tables | {CTX['focus']} focus ({CTX['focus_basis']}) | {CTX['excluded']} excluded | "
      f"access check={'on' if CTX['access_known'] else 'off'} | lineage={'on' if CTX['lineage_known'] else 'off'} ({round(time.time() - _t0, 1)}s)")
for n in INVENTORY_NOTES:
    print("  note:", n)

pillars = []
for p in PILLARS:
    _QUERY_LOG.clear()
    t0 = time.time()
    try:
        res = PROBES[p["key"]]()
    except Exception as e:
        res = unavailable(f"Could not be read: {short_err(e)}", reason_for(e))
    score = float(res["score"]) if res.get("available") else 0.0
    level = level_from_score(score)
    pillars.append({"key": p["key"], "name": p["name"], "weight": p["weight"], "score": round(score, 1),
                    "level": level, "level_label": LEVEL_LABELS[level], "available": bool(res.get("available")),
                    "unavailable_reason": res.get("unavailable_reason"), "note": res.get("note"),
                    "signals": res.get("signals", []), "gaps": res.get("gaps", []), "metrics": res.get("metrics", {}),
                    "drill_down": res.get("drill_down") or [], "source_queries": list(_QUERY_LOG),
                    "duration_s": round(time.time() - t0, 1)})
    print(f"{p['name']:<26} {pillars[-1]['score']:>5}  {'ok' if res.get('available') else 'UNAVAILABLE: ' + str(res.get('note'))}  ({pillars[-1]['duration_s']}s)")

overall = round(sum(p["score"] * p["weight"] for p in pillars) / sum(p["weight"] for p in pillars), 1)
ranked = sorted(pillars, key=lambda x: (x["score"], -x["weight"]))
stage = readiness_stage(overall)
summary = {
    "run_id": RUN_ID, "assessed_at": ASSESSED_AT.isoformat(), "assessed_by": ASSESSED_BY, "workspace_id": WORKSPACE_ID,
    "catalogs_requested": CATALOGS, "catalogs_assessed": READABLE, "catalogs_unreadable": UNREADABLE,
    "overall_score": overall, "level": level_from_score(overall), "level_label": LEVEL_LABELS[level_from_score(overall)],
    "readiness_stage": stage["label"], "readiness_detail": readiness_guidance(ranked, stage["detail"]),
    "top_gaps": [{"pillar": p["name"], "gap": g} for p in ranked for g in p["gaps"]][:6],
    "focus": CTX, "inventory_notes": list(INVENTORY_NOTES),
    "params": {"include_system_tables": INCLUDE_SYSTEM, "genie_source": GENIE_SOURCE, "adoption_scope": ADOPTION_SCOPE,
               "lookback_days": LOOKBACK_DAYS, "business_principals": BUSINESS_PRINCIPALS, "exclude_schema_pattern": EXCLUDE_SCHEMA_PATTERN,
               "gold_schema_pattern": GOLD_SCHEMA_PATTERN, "gold_table_pattern": GOLD_TABLE_PATTERN, "stale_days": STALE_DAYS,
               "output_mode": OUTPUT_MODE},
}
REPORT = {**summary, "pillars": pillars, "inventory_queries": INVENTORY_QUERIES}
print(f"\nOverall readiness: {overall} / 100 — {stage['label']} ({summary['level_label']})")

In [0]:
sq_measures, SQ_SCORE = [], None
if SEMANTIC_QUALITY:
    reset_llm()
    _QUERY_LOG.clear()
    _t0 = time.time()
    FOCUS_COLS = load_focus_columns()
    MVS, MV_FAILED = load_metric_views()
    SQ_SHARED_QUERIES = list(_QUERY_LOG)
    print(f"Loaded {len(FOCUS_COLS):,} focus/metric-view columns and {len(MVS)} metric-view definitions ({round(time.time() - _t0, 1)}s)")

    for d in SQ_DEFS:
        _QUERY_LOG.clear()
        t0 = time.time()
        try:
            res = SQ_PROBES[d["key"]]()
        except Exception as e:
            res = unavailable(f"Could not be assessed: {short_err(e)}", reason_for(e))
        score = float(res["score"]) if res.get("available") else 0.0
        level = level_from_score(score)
        sq_measures.append({"key": d["key"], "name": d["name"], "weight": d["weight"], "score": round(score, 1), "level": level,
                            "level_label": LEVEL_LABELS[level], "available": bool(res.get("available")),
                            "unavailable_reason": res.get("unavailable_reason"), "note": res.get("note"),
                            "signals": res.get("signals", []), "gaps": res.get("gaps", []), "metrics": res.get("metrics", {}),
                            "drill_down": res.get("drill_down") or [], "source_queries": list(_QUERY_LOG),
                            "duration_s": round(time.time() - t0, 1)})
        m = sq_measures[-1]
        print(f"{m['name']:<26} {m['score']:>5}  {'ok' if m['available'] else 'UNAVAILABLE: ' + str(m['note'])}  ({m['duration_s']}s)")

    # Unlike coverage, unavailable quality checks are left out and the rest re-weighted (the LLM step is optional).
    avail = [m for m in sq_measures if m["available"]]
    if avail:
        SQ_SCORE = round(sum(m["score"] * m["weight"] for m in avail) / sum(m["weight"] for m in avail), 1)
    sq_ranked = sorted(avail, key=lambda x: (x["score"], -x["weight"]))
    summary["semantic_quality"] = {
        "score": SQ_SCORE, "level_label": LEVEL_LABELS[level_from_score(SQ_SCORE)] if SQ_SCORE is not None else None,
        "measures_available": len(avail), "measures_total": len(sq_measures), "llm_endpoint": _LLM.get("endpoint"),
        "top_gaps": [{"pillar": m["name"], "gap": g} for m in sq_ranked for g in m["gaps"]][:6]}
    REPORT.update(summary)
    REPORT["semantic_quality_measures"] = sq_measures
    REPORT["semantic_quality_queries"] = SQ_SHARED_QUERIES
    print(f"\nSemantic quality: {SQ_SCORE if SQ_SCORE is not None else 'n/a'} / 100 "
          f"({len(avail)} of {len(sq_measures)} checks available; LLM: {_LLM.get('endpoint') or 'off - ' + str(_LLM.get('why'))})")
else:
    summary["semantic_quality"] = None
    REPORT.update(summary)
    print("Semantic quality checks are off (semantic_quality = false).")

In [0]:
def _num(v):
    """937 -> '937', 32982 -> '32,982', 100.0 -> '100', 31.8 -> '31.8'."""
    if isinstance(v, bool) or v is None:
        return str(v)
    if isinstance(v, (int, float)):
        return f"{v:,.0f}" if float(v).is_integer() else f"{v:,.1f}"
    return str(v)

def _fmt_val(s):
    u, v = s.get("unit") or "", s["value"]
    if isinstance(v, bool) or not isinstance(v, (int, float)):
        return str(v)   # e.g. 'n/a' or '3 / 12': no unit
    return _num(v) + (u if u == "%" else (" " + u if u else ""))

def _fmt_sig(s):
    return f"{s['label']}: {_fmt_val(s)}"

def _mismatch(p):
    """Strong score but gaps remain."""
    return p["available"] and p["score"] >= 65 and len(p["gaps"]) > 0

MISMATCH_HINT = "The score is strong, but the gaps below are still worth closing."

REPORT_CSS = """
.gr{font-family:-apple-system,BlinkMacSystemFont,'Segoe UI',Helvetica,Arial,sans-serif;color:#1b3139;font-size:13px;line-height:1.45;max-width:1180px}
.gr *{box-sizing:border-box}
.gr .hero{display:flex;gap:28px;align-items:center;background:#1b3139;color:#fff;border-radius:12px;padding:22px 26px}
.gr .rings{display:flex;gap:20px;flex:none}
.gr .ring{border-radius:50%;display:flex;align-items:center;justify-content:center;margin:0 auto}
.gr .ring>div{border-radius:50%;background:#1b3139;display:flex;flex-direction:column;align-items:center;justify-content:center}
.gr .ring b{font-size:28px;line-height:1}.gr .ring span{font-size:11px;opacity:.7}
.gr .eyebrow{text-transform:uppercase;letter-spacing:.08em;font-size:11px;opacity:.65}
.gr h1{font-size:22px;margin:2px 0 6px;font-weight:600}
.gr .meta{font-size:12px;opacity:.7;margin-top:6px}
.gr .pill{display:inline-block;border-radius:999px;padding:2px 10px;font-size:11px;font-weight:600}
.gr .chip{display:inline-block;border-radius:6px;padding:1px 7px;font-size:11px;background:#eef1f2;color:#4a5b62;margin-left:6px;font-weight:500}
.gr h2{font-size:15px;margin:26px 0 4px;font-weight:600}
.gr .lede{color:#6b7c83;font-size:12px;margin:0 0 10px}
.gr .cards{display:grid;grid-template-columns:repeat(auto-fill,minmax(150px,1fr));gap:10px}
.gr .card{border:1px solid #e3e8ea;border-radius:10px;padding:12px;background:#fff}
.gr .card .nm{font-size:12px;font-weight:600;min-height:32px}
.gr .card .sc{font-size:24px;font-weight:700;margin:4px 0}
.gr .bar{height:6px;background:#eef1f2;border-radius:3px;overflow:hidden}.gr .bar>div{height:100%;border-radius:3px}
.gr .sub{font-size:11px;color:#6b7c83;margin-top:6px}
.gr .warn{color:#b26b00;font-weight:600}
.gr ol.gaps{margin:0;padding:0;list-style:none;counter-reset:g}
.gr ol.gaps li{counter-increment:g;display:flex;gap:10px;padding:8px 0;border-bottom:1px solid #eef1f2}
.gr ol.gaps li:before{content:counter(g);flex:none;width:22px;height:22px;border-radius:50%;background:#ff3621;color:#fff;font-size:11px;font-weight:700;display:flex;align-items:center;justify-content:center}
.gr ol.gaps.q li:before{background:#4462c9}
.gr .pillar{border:1px solid #e3e8ea;border-radius:10px;padding:16px 18px;margin-bottom:12px;background:#fff}
.gr .ph{display:flex;align-items:center;gap:10px;margin-bottom:8px}
.gr .ph .t{font-size:15px;font-weight:600;flex:1}
.gr .ph .s{font-size:20px;font-weight:700}
.gr .tiles{display:grid;grid-template-columns:repeat(auto-fill,minmax(150px,1fr));gap:8px;margin:12px 0}
.gr .tile{background:#f7f9fa;border-radius:8px;padding:8px 10px}
.gr .tile b{font-size:17px;display:block}.gr .tile span{font-size:11px;color:#6b7c83}
.gr .glist{margin:6px 0 0;padding:0;list-style:none}
.gr .glist li{border-left:3px solid #ffab00;background:#fff8eb;padding:6px 10px;margin-bottom:6px;border-radius:0 6px 6px 0}
.gr .ok{color:#00805a;font-size:12px}
.gr .note{font-size:11px;color:#6b7c83;margin-top:8px}
.gr .flag{background:#fff3e0;color:#8a5300;border-radius:6px;padding:6px 10px;font-size:12px;margin:8px 0}
.gr .na{opacity:.6}
"""

def _color(x):
    return "#00a972" if x >= 65 else "#ffab00" if x >= 40 else "#ff3621"

def _level_pill(p):
    c = _color(p["score"]) if p.get("available", True) else "#9aa7ad"
    label = p["level_label"] if p.get("available", True) else "Unavailable"
    return f"<span class='pill' style='background:{c}22;color:{c}'>{html.escape(label)}</span>"

def _scope_chip(p):
    scope = (p.get("metrics") or {}).get("scope")
    return f"<span class='chip'>{'workspace-wide' if scope == 'workspace' else 'catalog'}</span>" if scope else ""

def _ring(score, label, size):
    inner = size - 24
    return (f"<div style='text-align:center'><div class='ring' style='width:{size}px;height:{size}px;"
            f"background:conic-gradient({_color(score)} {score * 3.6}deg, #ffffff22 0)'>"
            f"<div style='width:{inner}px;height:{inner}px'><b>{_num(score)}</b><span>/ 100</span></div></div>"
            f"<div class='eyebrow' style='margin-top:6px'>{html.escape(label)}</div></div>")

def _cards_html(items):
    e, out = html.escape, ""
    for p in items:
        if not p["available"]:
            sub = ("<div class='sub'>Not run (switched off)</div>" if p["unavailable_reason"] == "not_enabled" else
                   f"<div class='sub'>Unavailable: {e(p['unavailable_reason'] or 'not read')}</div>")
        elif _mismatch(p):
            sub = f"<div class='sub warn' title='{e(MISMATCH_HINT)}'>Strong score, {len(p['gaps'])} open gap(s)</div>"
        elif p["gaps"]:
            sub = f"<div class='sub'>{len(p['gaps'])} gap(s)</div>"
        else:
            sub = "<div class='sub'>No gaps</div>"
        out += (f"<div class='card{' na' if not p['available'] else ''}'><div class='nm'>{e(p['name'])}</div>"
                f"<div class='sc' style='color:{_color(p['score'])}'>{_num(p['score']) if p['available'] else '—'}</div>"
                f"<div class='bar'><div style='width:{p['score']}%;background:{_color(p['score'])}'></div></div>"
                f"<div class='sub'>weight {p['weight']} · {e(p['level_label'] if p['available'] else 'Unavailable')}</div>{sub}</div>")
    return f"<div class='cards'>{out}</div>"

def _gaps_html(gaps, cls=""):
    e = html.escape
    items = "".join(f"<li><div><span class='chip' style='margin:0 6px 0 0'>{e(g['pillar'])}</span>{e(g['gap'])}</div></li>" for g in gaps)
    return f"<ol class='gaps {cls}'>{items or '<li><div>No gaps found.</div></li>'}</ol>"

def _details_html(items):
    e, out = html.escape, ""
    for p in items:
        tiles = "".join(f"<div class='tile' title='{e(s.get('detail') or '')}'><b>{e(_fmt_val(s))}</b><span>{e(s['label'])}</span></div>"
                        for s in p["signals"])
        if p["gaps"]:
            glist = "<ul class='glist'>" + "".join(f"<li>{e(g)}</li>" for g in p["gaps"]) + "</ul>"
        else:
            glist = "<div class='ok'>No gaps found.</div>" if p["available"] else ""
        flag = f"<div class='flag'>{e(MISMATCH_HINT)}</div>" if _mismatch(p) else ""
        note = f"<div class='note'>{e(p['note'])}</div>" if p["note"] else ""
        score_txt = _num(p["score"]) if p["available"] else "—"
        out += (f"<div class='pillar{' na' if not p['available'] else ''}'>"
                f"<div class='ph'><div class='t'>{e(p['name'])}<span class='chip'>weight {p['weight']}</span>{_scope_chip(p)}</div>"
                f"{_level_pill(p)}<div class='s' style='color:{_color(p['score'])}'>{score_txt}</div></div>"
                f"<div class='bar'><div style='width:{p['score']}%;background:{_color(p['score'])}'></div></div>"
                f"{flag}<div class='tiles'>{tiles}</div>{glist}{note}</div>")
    return out

def build_html(r):
    e = html.escape
    sq = r.get("semantic_quality") or {}
    sq_items = r.get("semantic_quality_measures") or []
    rings = _ring(r["overall_score"], "Coverage", 118)
    if sq.get("score") is not None:
        rings += _ring(sq["score"], "Semantic quality", 118)
    unread = (f"<div class='meta'>Skipped (not readable): {e(', '.join(r['catalogs_unreadable']))}</div>"
              if r.get("catalogs_unreadable") else "")
    notes_html = "".join("<div class='meta'>Note: " + e(n) + "</div>" for n in r.get("inventory_notes", []))
    focus = r["focus"]
    sq_meta = ""
    if sq:
        sq_meta = (f"<div class='meta'>Semantic quality: {sq['measures_available']} of {sq['measures_total']} checks ran"
                   f"{'; LLM ' + e(sq['llm_endpoint']) if sq.get('llm_endpoint') else '; LLM checks off'}.</div>")
    sq_section = ""
    if sq_items:
        sq_section = ("<h2>Semantic quality</h2><p class='lede'>Is the documentation good enough for Genie to answer correctly? "
                      "Scored separately from coverage; checks that couldn't run are left out and the rest re-weighted.</p>"
                      f"{_cards_html(sq_items)}<h2>Top semantic quality gaps</h2>{_gaps_html(sq.get('top_gaps', []), 'q')}")
    sq_detail = f"<h2>Semantic quality detail</h2>{_details_html(sq_items)}" if sq_items else ""
    return f"""<style>{REPORT_CSS}</style><div class='gr'>
      <div class='hero'>
        <div class='rings'>{rings}</div>
        <div style='flex:1'>
          <div class='eyebrow'>Genie Ontology Readiness</div>
          <h1>{e(', '.join(r['catalogs_assessed']))}</h1>
          <span class='pill' style='background:{_color(r['overall_score'])};color:#fff'>{e(r['readiness_stage'])}</span>
          <span class='pill' style='background:#ffffff22;color:#fff'>{e(r['level_label'])}</span>
          <div style='margin-top:10px'>{e(r['readiness_detail'])}</div>
          <div class='meta'>Coverage scored on {focus['focus']:,} focus tables ({e(focus['focus_basis'])}) of {focus['tables']:,}; {focus['excluded']:,} excluded by schema pattern.</div>
          {sq_meta}
          <div class='meta'>Run {e(r['run_id'])} · {e(r['assessed_at'][:16].replace('T', ' '))} UTC · {e(r['assessed_by'])}</div>{unread}
          {notes_html}
        </div>
      </div>
      <h2>Coverage pillars</h2><p class='lede'>How much of the focus set has the foundations Genie relies on.</p>
      {_cards_html(r['pillars'])}
      <h2>Top coverage gaps</h2>{_gaps_html(r['top_gaps'])}
      {sq_section}
      <h2>Coverage detail</h2>{_details_html(r['pillars'])}
      {sq_detail}
    </div>"""

def _md_section(items):
    out = []
    for p in items:
        out += ["", f"### {p['name']} ({_num(p['score']) if p['available'] else 'unavailable'})", ""]
        out += [f"* {_fmt_sig(s)} ({s['detail']})" for s in p["signals"]]
        out += [f"* Gap: {g}" for g in p["gaps"]] + ([f"* Note: {p['note']}"] if p["note"] else [])
    return out

def build_markdown(r):
    sq = r.get("semantic_quality") or {}
    sq_items = r.get("semantic_quality_measures") or []
    head = f"**Coverage: {r['overall_score']} / 100 ({r['readiness_stage']}, {r['level_label']})**"
    if sq.get("score") is not None:
        head += f" | **Semantic quality: {sq['score']} / 100 ({sq['level_label']})**"
    out = [f"# Genie Ontology Readiness: {', '.join(r['catalogs_assessed'])}", "",
           f"Run `{r['run_id']}` | {r['assessed_at'][:19]} UTC | by {r['assessed_by']}", "", head, "", r["readiness_detail"], "",
           f"Coverage scored on {r['focus']['focus']} focus tables ({r['focus']['focus_basis']}) of {r['focus']['tables']}; "
           f"{r['focus']['excluded']} excluded by schema pattern.", "",
           "| Axis | Measure | Weight | Score | Level |", "| --- | --- | --- | --- | --- |"]
    for axis, items in (("Coverage", r["pillars"]), ("Quality", sq_items)):
        out += [f"| {axis} | {p['name']} | {p['weight']} | {_num(p['score']) if p['available'] else 'n/a'} | "
                f"{p['level_label'] if p['available'] else 'Unavailable'} |" for p in items]
    out += ["", "## Top coverage gaps", ""] + [f"1. **{g['pillar']}**: {g['gap']}" for g in r["top_gaps"]]
    if sq_items:
        out += ["", "## Top semantic quality gaps", ""] + [f"1. **{g['pillar']}**: {g['gap']}" for g in sq.get("top_gaps", [])]
    out += ["", "## Coverage detail"] + _md_section(r["pillars"])
    if sq_items:
        out += ["", "## Semantic quality detail"] + _md_section(sq_items)
    return "\n".join(out) + "\n"

ALL_MEASURES = [("coverage", p) for p in pillars] + [("quality", m) for m in sq_measures]
REPORT_HTML, REPORT_MD = build_html(REPORT), build_markdown(REPORT)
DRILL_ROWS = [{"pillar": p["key"], **d} for _, p in ALL_MEASURES for d in p["drill_down"]]

displayHTML(REPORT_HTML)

In [0]:
# Focus tables: one row per table with every readiness flag, worst first.
focus_view = INV.loc[INV["focus"], INV_COLS].copy()
focus_view["issues"] = ((focus_view["comment_ok"] == 0).astype(int) + (focus_view["col_meaningful_pct"] < 60).astype(int)
                        + focus_view["domain"].isna().astype(int) + (~focus_view["certified"]).astype(int)
                        + (~focus_view["stewarded"]).astype(int) + (focus_view["fresh"] == 0).astype(int))
print(f"--- Focus tables ({len(focus_view)}) ---")
focus_view["domain"] = focus_view["domain"].fillna("")
display(focus_view.sort_values(["issues", "human_users_90d"], ascending=[False, False]).drop(columns=["fq"]).astype(str))

# One table per pillar / quality check showing where the gaps sit (worst first where it applies).
for _axis, p in ALL_MEASURES:
    if p["drill_down"]:
        print(f"--- {p['name']} ({_axis}) ---")
        display(pd.DataFrame(p["drill_down"]).astype(str))

In [0]:
# Per-table inventory with plain Python values (None for missing) for the Delta write.
INV_EXPORT = INV.loc[:, INV_COLS].astype(object).where(INV.loc[:, INV_COLS].notna(), None)
for _c in ("comment_ok", "n_cols", "n_cols_meaningful", "masked_cols", "fresh", "human_users_90d", "sp_users_90d", "human_reads_lb"):
    INV_EXPORT[_c] = INV_EXPORT[_c].map(lambda v: None if v is None else int(v))
for _c in ("focus", "is_gold", "top_accessed", "is_metric_view", "excluded", "junk_name", "dup_name", "has_pk", "fk_connected",
           "any_tag", "stewarded", "certified", "deprecated", "business_access", "row_filter"):
    INV_EXPORT[_c] = INV_EXPORT[_c].map(lambda v: None if v is None else bool(v))
INV_EXPORT["col_meaningful_pct"] = INV_EXPORT["col_meaningful_pct"].map(lambda v: None if v is None else float(v))


def write_tables(schema: str) -> str:
    parts = [x.strip().strip("`") for x in schema.split(".")]
    if len(parts) != 2:
        raise ValueError("output_schema must be catalog.schema")
    fq = f"{qi(parts[0])}.{qi(parts[1])}"
    catalogs = ",".join(READABLE)
    runs = spark.createDataFrame([(RUN_ID, ASSESSED_AT, ASSESSED_BY, WORKSPACE_ID, catalogs, float(overall), summary["level"],
                                   summary["level_label"], summary["readiness_stage"], summary["readiness_detail"],
                                   json.dumps(summary["top_gaps"]), json.dumps(summary["params"]), json.dumps(REPORT, default=str),
                                   None if SQ_SCORE is None else float(SQ_SCORE))],
        "run_id string, assessed_at timestamp, assessed_by string, workspace_id string, catalogs string, overall_score double, "
        "level int, level_label string, readiness_stage string, readiness_detail string, top_gaps_json string, params_json string, report_json string, "
        "semantic_quality_score double")
    pil = spark.createDataFrame([(RUN_ID, ASSESSED_AT, catalogs, p["key"], p["name"], p["weight"], float(p["score"]), p["level"],
                                  p["level_label"], p["available"], p["unavailable_reason"], p["note"], json.dumps(p["signals"], default=str),
                                  p["gaps"], json.dumps(p["metrics"], default=str), p["source_queries"], axis) for axis, p in ALL_MEASURES],
        "run_id string, assessed_at timestamp, catalogs string, pillar_key string, pillar_name string, weight int, score double, level int, "
        "level_label string, available boolean, unavailable_reason string, note string, signals_json string, gaps array<string>, "
        "metrics_json string, source_queries array<string>, axis string")
    drill = spark.createDataFrame([(RUN_ID, d["pillar"], d.get("catalog"), d.get("schema"), json.dumps(d, default=str)) for d in DRILL_ROWS]
                                  or [(RUN_ID, None, None, None, None)],
        "run_id string, pillar_key string, catalog string, schema string, row_json string").where("pillar_key IS NOT NULL")
    tables = spark.createDataFrame(
        [(RUN_ID, ASSESSED_AT, *row) for row in INV_EXPORT.values.tolist()],
        "run_id string, assessed_at timestamp, fq string, catalog string, schema string, table_name string, table_type string, owner string, "
        "owner_type string, focus boolean, is_gold boolean, top_accessed boolean, is_metric_view boolean, excluded boolean, junk_name boolean, "
        "dup_name boolean, comment_ok int, n_cols int, n_cols_meaningful int, col_meaningful_pct double, has_pk boolean, fk_connected boolean, "
        "domain string, any_tag boolean, stewarded boolean, certified boolean, deprecated boolean, business_access boolean, row_filter boolean, "
        "masked_cols int, fresh int, last_altered string, human_users_90d int, sp_users_90d int, human_reads_lb int")
    for name, df in (("genie_readiness_runs", runs), ("genie_readiness_pillars", pil), ("genie_readiness_drilldown", drill),
                     ("genie_readiness_tables", tables)):
        df.write.mode("append").option("mergeSchema", "true").saveAsTable(f"{fq}.{qi(name)}")
    return f"{parts[0]}.{parts[1]}.genie_readiness_runs / _pillars / _drilldown / _tables"


def write_files(base_dir: str) -> str:
    d = f"{base_dir}/{RUN_ID}"
    os.makedirs(d, exist_ok=True)
    with open(f"{d}/report.json", "w") as f: json.dump(REPORT, f, indent=2, default=str)
    with open(f"{d}/report.md", "w") as f: f.write(REPORT_MD)
    with open(f"{d}/report.html", "w") as f: f.write("<html><body>" + REPORT_HTML + "</body></html>")
    keys = sorted({k for r in DRILL_ROWS for k in r}, key=lambda k: (k not in ("pillar", "catalog", "schema"), k))
    with open(f"{d}/drilldown.csv", "w", newline="") as f:
        w = csv.DictWriter(f, fieldnames=keys or ["pillar"]); w.writeheader(); w.writerows(DRILL_ROWS)
    INV.loc[:, INV_COLS].to_csv(f"{d}/tables.csv", index=False)
    return d


def ws_dir() -> str:
    p = OUTPUT_WORKSPACE_DIR or f"/Users/{ASSESSED_BY}/genie_readiness_runs"
    return p if p.startswith("/Workspace") else "/Workspace" + (p if p.startswith("/") else "/" + p)


def _valid_volume(p):
    """Must be /Volumes/<catalog>/<schema>/<volume>[/...]; anything else would land on the driver's local disk."""
    return bool(re.match(r"^/Volumes/[^/]+/[^/]+/[^/]+", p or ""))

written, failed = [], []
def attempt(label, fn, arg):
    try:
        written.append(f"{label}: {fn(arg)}")
    except Exception as e:
        failed.append(f"{label}: {short_err(e)}")

if OUTPUT_MODE == "table":
    if OUTPUT_SCHEMA: attempt("Delta tables", write_tables, OUTPUT_SCHEMA)
    else: failed.append("table: set output_schema")
elif OUTPUT_MODE == "volume":
    if _valid_volume(OUTPUT_VOLUME_PATH): attempt("Volume", write_files, OUTPUT_VOLUME_PATH.rstrip("/"))
    else: failed.append("volume: set output_volume_path to /Volumes/<catalog>/<schema>/<volume>")
elif OUTPUT_MODE == "workspace":
    attempt("Workspace", write_files, ws_dir())
elif OUTPUT_MODE == "auto":
    if OUTPUT_SCHEMA: attempt("Delta tables", write_tables, OUTPUT_SCHEMA)
    if OUTPUT_VOLUME_PATH:
        if _valid_volume(OUTPUT_VOLUME_PATH): attempt("Volume", write_files, OUTPUT_VOLUME_PATH.rstrip("/"))
        else: failed.append(f"Volume: '{OUTPUT_VOLUME_PATH}' is not /Volumes/<catalog>/<schema>/<volume>")
    if not written: attempt("Workspace", write_files, ws_dir())

for x in written: print("Saved   ->", x)
for x in failed:  print("Skipped ->", x)
if OUTPUT_MODE == "display_only" or not written:
    if OUTPUT_MODE != "display_only":
        print("\nNo output location was writable. Copy the JSON below, or export this notebook (File > Export > HTML).")
    print(json.dumps(REPORT, indent=2, default=str))